# TC-WPN — Phase 6: error analysis and root-cause diagnosis

**Component:** R26-DS-012 / TC-WPN — Dulhara Kaushalya (IT22130648)
**Supervisor instruction this notebook implements, verbatim:**

> "මුලින් බලන්න හරි clinical notes මෙන්න මේ ටික නම් හරියටම recognize කරනවා. මෙන්න මේවා recognize කරන්නේ නෑ."
> — first identify which clinical notes are classified correctly and which are classified incorrectly.

> "ඒ නෝට්ස් අර නෝට්ස් වලට වඩා මොකක්ද මේ තියෙන වෙනස"
> — what is different about the incorrectly classified notes compared with the correctly classified ones?

> "embeddings ටික අරගෙන වෙන චෙක් කරලා බැලුවද" — have you taken the embeddings and checked them separately? (Answer given in the meeting: **no**. This notebook is where that gets done.)

---

## What this notebook is

Phases 1–5 answered **"are the TC-WPN weighting mechanisms doing anything?"** — yes, they are active
(`H_norm` 0.943, max/min 19.4, corr(w, days) −0.631 for `tcwpn_full`) but they do not improve
discrimination (paired Δ = +0.0006, 1/5 seeds better, p = 0.886).

That is **not** the analysis the supervisor has now requested. He asked for **root cause first**:

```text
Current model
     ↓
Find correctly classified samples
Find incorrectly classified samples
     ↓
Study the misclassified clinical notes
     ↓
Ask: what makes these notes difficult?
     ↓
       ┌───────────────┬─────────────────┐
       ↓               ↓                 ↓
   DATA problem    EMBEDDING problem   MODEL problem
       ↓               ↓                 ↓
   rare note type   poor separation    prototype/
   insufficient     poor semantic      temporal/
   examples         representation     weighting issue
       ↓               ↓                 ↓
   Fix data         investigate BERT   investigate TC-WPN
       └───────────────┴─────────────────┘
                       ↓
                 retrain/evaluate
                       ↓
              compare against baseline
```

## What this notebook is NOT

It does **not** change `model.py`, `train.py`, `sampler.py`, or any config. Nothing is retrained
in sections 1–17. The forbidden pattern is:

```text
0.737 → change BERT → 0.741 → change LR → 0.744 → change dropout → 0.739
```

The required pattern is:

```text
0.737 → diagnose errors → find root cause → form hypothesis
      → make ONE targeted change → retrain → test → compare
```

## Frozen baseline this notebook analyses (do not re-select anything)

Five-seed, patient-disjoint, ICD-labelled, zero-leakage-certificate benchmark, already in the repo:

| config | mean AUROC | SD | paired Δ vs `aux_only` | seeds better | p |
|---|---:|---:|---:|---:|---:|
| `aux_only` | 0.7371 | 0.0081 | — | — | — |
| `temporal_aux` | 0.7377 | 0.0032 | +0.0006 | 2/5 | 0.906 |
| `pcw_aux` | 0.7291 | 0.0083 | −0.0080 | 1/5 | 0.150 |
| `tcwpn_full` | 0.7377 | 0.0031 | +0.0006 | 1/5 | 0.886 |

The run dissected below is `tcwpn_full_k5_seed42` (test AUROC 0.7379, 2 278 patients,
1 353 case / 925 control, locked threshold 0.26931 chosen on validation only).

**The 0.80 in the proposal is not a target.** If a leakage-controlled, root-caused intervention
reaches it, good. If not, the measured number is reported.

## Section map — exactly the 20 steps requested

```text
 0. Train / validation / test performance   (feedback §18 — not in the 20-step list, but asked for)
 1. Load frozen TC-WPN test predictions
 2. Identify TP / TN / FP / FN
 3. Link predictions to patient notes
 4. Build note-level error table
 5. Analyze note length
 6. Analyze note type
 7. Analyze temporal distance
 8. Analyze clinical-text characteristics
 9. Compare correct vs incorrect groups
10. Extract ClinicalBERT embeddings
11. Measure embedding separation
12. Visualize embedding space
13. Analyze FP/FN embedding neighborhoods
14. Inspect prototype distances
15. Inspect support weights for errors
16. Determine likely root cause
17. Form one hypothesis
18. Only then modify the model/data
19. Retrain
20. Evaluate against frozen baseline
```

## Inputs to attach before running

| input | why |
|---|---|
| Stage A notebook output | `pkl/`, `plans/`, and `cohort_psych_mimic4idx.csv` — **the cohort CSV is the only place the note text lives**; the pkl holds token ids only |
| Phase 3B `tcwpn_full` output | `best.pt`, `manifest.json`, `predictions_test.csv` (all three are gitignored, so the repo clone cannot supply them) |
| Phase 3B `aux_only` output | the auxiliary-controlled reference |

Accelerator: **GPU T4**. Sections 0 and 10–15 run the encoder; everything else is CPU.

> **MIMIC-IV DUA:** this notebook prints note excerpts for inspection. Keep them inside the Kaggle
> session. Do not paste note text into the paper, slides, GitHub, or any shared document.


In [ ]:
# ---------------------------------------------------------------------------
# 0.0  Repo + dependencies.  Internet must be ON (Settings -> Internet).
#      Nothing here modifies model.py, train.py, sampler.py or any config.
# ---------------------------------------------------------------------------
!rm -rf /kaggle/working/tcwpn_test
!git clone -q https://github.com/dulhara79/tcwpn_test.git /kaggle/working/tcwpn_test
%cd /kaggle/working/tcwpn_test
!git log --oneline -1
!pip install -q -r requirements.txt 2>&1 | tail -2

import os, sys
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONPATH"] = "src"
sys.path.insert(0, "/kaggle/working/tcwpn_test/src")

import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
print("ready")

In [ ]:
# ---------------------------------------------------------------------------
# 0.1  Locate every input.  Fails loudly rather than analysing the wrong file.
# ---------------------------------------------------------------------------
import glob, json, shutil
from pathlib import Path
import numpy as np
import pandas as pd

STEM, K, SEED = "psych_mimic4idx", 5, 42
RUN_NAME  = f"tcwpn_full_k{K}_seed{SEED}"   # the run under investigation
BASE_NAME = f"aux_only_k{K}_seed{SEED}"     # auxiliary-controlled reference

OUT = Path("/kaggle/working/phase6"); OUT.mkdir(parents=True, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ---- Stage A dataset: pkl/ + plans/ side by side ---------------------------
stage_a = next((p.parent for p in Path("/kaggle/input").rglob("plans")
                if (p.parent / "pkl").exists()), None)
if stage_a is None:
    raise SystemExit(
        "Stage A dataset not found. + Add Input -> Your Work -> Notebook Output "
        "-> kaggle_stage_a_data_prep")
PKL_DIR, PLAN_DIR = stage_a / "pkl", stage_a / "plans"

# ---- the cohort CSV is the ONLY carrier of note text -----------------------
hits = sorted(Path("/kaggle/input").rglob(f"cohort_{STEM}.csv"))
if not hits:
    raise SystemExit(
        f"cohort_{STEM}.csv not found under /kaggle/input. The pkl files store "
        f"token ids only, so without this file sections 3-9 cannot run.")
COHORT_CSV = hits[0]

# ---- import the two run directories (best.pt is gitignored) ----------------
RESULTS = Path("/kaggle/working/results") / STEM

def import_run(name):
    found = glob.glob(f"/kaggle/input/**/{name}/manifest.json", recursive=True)
    if not found:
        return None
    src = Path(sorted(found)[0]).parent
    dst = RESULTS / name
    dst.mkdir(parents=True, exist_ok=True)
    for f in src.iterdir():
        if f.is_file():
            shutil.copy2(f, dst)
    return dst

RUN_DIR  = import_run(RUN_NAME)
BASE_DIR = import_run(BASE_NAME)
if RUN_DIR is None:
    raise SystemExit(f"{RUN_NAME} not found. Add its Phase 3B notebook output as an input.")

print("stage A     :", stage_a)
print("cohort CSV  :", COHORT_CSV)
print("run dir     :", RUN_DIR, sorted(p.name for p in RUN_DIR.iterdir()))
print("baseline dir:", BASE_DIR if BASE_DIR else "NOT FOUND (section 20 will be limited)")

## 0. Train / validation / test performance

Feedback **§18**: *"Your supervisor talked about training vs testing accuracy. You told him you
hadn't properly separated them."*

Reported here on all three splits, and **not on accuracy alone** — for a 59% -prevalence binary
clinical task accuracy is close to uninformative. AUROC, PR-AUC, sensitivity, specificity, F1,
Brier and ECE all come from the repo's own `tcwpn.metrics.compute_metrics`, at the threshold
locked on validation by `train.py`.

Every split is scored through the **frozen episode plan** for that split, so the numbers are
comparable to each other and to `eval_test.json` already in the repo.

* `test`, `val` — the full coverage-guaranteed eval plans (every patient queried 3×).
* `train` — the training plan is a random 3 000-episode plan, so it is capped here at
  `TRAIN_EPISODES` episodes. That gives a **discrimination estimate on the training split**, not a
  coverage-complete number. Say exactly that when reporting it.

This is the underfitting / overfitting / intrinsically-overlapping test:

| pattern | reading |
|---|---|
| train ≫ val ≈ test | overfitting |
| train ≈ val ≈ test, all modest | underfitting **or** an intrinsically overlapping problem |
| train ≈ val ≈ test, all high | fine |

Sections 10–16 are what separate the two entries in the middle row.

In [ ]:
# ---------------------------------------------------------------------------
# 0.2  Shared machinery: embed once per record, then reuse.
#      Replicates ClinicalEmbedder.forward exactly (bert -> [CLS] -> mean over
#      chunks -> projection) but ALSO returns the pre-projection pooled [CLS],
#      because section 11 needs to know whether a separation problem lives in
#      the encoder or is introduced by the 256-d projection head.
# ---------------------------------------------------------------------------
import torch.nn.functional as Fn
from tqdm.auto import tqdm

from tcwpn.model import build_model
from tcwpn.sampler import RecordStore, EpisodePlan, store_fingerprint
from tcwpn.metrics import compute_metrics


def load_run(run_dir, device=DEVICE):
    manifest = json.loads((Path(run_dir) / "manifest.json").read_text())
    model = build_model(manifest["config"]["model"]).to(device)
    ckpt = torch.load(Path(run_dir) / "best.pt", map_location=device)
    model.load_state_dict(ckpt["model"])
    model.eval()
    return model, manifest


def episode_slots(ep):
    '''(record_index, class) lists in the SAME order collate_episode uses.'''
    sup, qry = [], []
    for c in sorted(ep["support"], key=int):
        sup += [(int(i), int(c)) for i in ep["support"][c]]
    for c in sorted(ep["query"], key=int):
        qry += [(int(i), int(c)) for i in ep["query"][c]]
    return sup, qry


@torch.no_grad()
def embed_records(model, store, indices, batch_size=32, desc="embedding"):
    '''
    indices -> (row_of, cls_768, proj_256) where row_of maps a record index to a
    row in the returned matrices. Chunks of a note are mean-pooled exactly as
    ClinicalEmbedder does.
    '''
    indices = sorted(set(int(i) for i in indices))
    row_of = {ix: r for r, ix in enumerate(indices)}

    rows_ids, rows_mask, note_row = [], [], []
    for ix in indices:
        r = store.records[ix]
        for cid, cmask in zip(r["input_ids"], r["attention_mask"]):
            rows_ids.append(cid); rows_mask.append(cmask); note_row.append(row_of[ix])

    H = model.embedder.hidden_size
    cls_sum = torch.zeros(len(indices), H, device=DEVICE)
    counts  = torch.zeros(len(indices), 1, device=DEVICE)
    for s in tqdm(range(0, len(rows_ids), batch_size), desc=desc, leave=False):
        ids  = torch.tensor(rows_ids[s:s + batch_size],  dtype=torch.long, device=DEVICE)
        mask = torch.tensor(rows_mask[s:s + batch_size], dtype=torch.long, device=DEVICE)
        nidx = torch.tensor(note_row[s:s + batch_size],  dtype=torch.long, device=DEVICE)
        cls = model.embedder.bert(input_ids=ids, attention_mask=mask).last_hidden_state[:, 0, :]
        cls_sum.index_add_(0, nidx, cls.to(cls_sum.dtype))
        counts.index_add_(0, nidx, torch.ones(len(nidx), 1, device=DEVICE))
    pooled = cls_sum / counts.clamp(min=1.0)
    proj = model.embedder.projection(pooled)
    return row_of, pooled, proj


@torch.no_grad()
def score_plan_cached(model, store, plan, max_episodes=None, batch_size=32,
                      desc="scoring", want_geometry=False):
    '''
    Re-implements PrototypicalModel.forward on cached embeddings so the encoder
    runs once per record instead of once per episode. Returns a per-query-note
    DataFrame. With want_geometry=True it also returns prototype distances,
    support-weight statistics, and the uniform-weight counterfactual.

    Verified against predictions_test.csv in section 1 -- if the max absolute
    difference there is not ~1e-6 this function is wrong and nothing below it
    can be trusted.
    '''
    eps = list(plan)[:max_episodes] if max_episodes else list(plan)
    needed = set()
    for ep in eps:
        s, q = episode_slots(ep)
        needed |= {i for i, _ in s} | {i for i, _ in q}
    row_of, pooled, proj = embed_records(model, store, needed, batch_size, desc=f"{desc}: encode")

    days_all = [float(r.get("days_before_patient_last_note", 0.0)) for r in store.records]
    rows = []
    for ep_i, ep in enumerate(tqdm(eps, desc=f"{desc}: episodes", leave=False)):
        sup, qry = episode_slots(ep)
        classes = sorted({c for _, c in sup})
        cls_to_col = {c: i for i, c in enumerate(classes)}

        protos, protos_u, wstat = [], [], {}
        for c in classes:
            idxs = [i for i, cc in sup if cc == c]
            E = proj[[row_of[i] for i in idxs]]
            d = torch.tensor([days_all[i] for i in idxs], dtype=torch.float32, device=DEVICE)
            p, w = model.build_prototype(E, d)
            protos.append(p)
            protos_u.append(Fn.normalize(E.mean(0), dim=-1))       # weights forced to 1/K
            wv = w.detach().float().cpu().numpy().ravel()
            wv = wv / max(wv.sum(), 1e-12)
            wstat[c] = (wv, np.array([days_all[i] for i in idxs], dtype=float))

        qidx = [i for i, _ in qry]
        QE = proj[[row_of[i] for i in qidx]]
        logits = model.classify(QE, protos)
        probs = Fn.softmax(logits, dim=-1)
        pos_col = cls_to_col.get(1, logits.size(1) - 1)
        p_anx = probs[:, pos_col].detach().float().cpu().numpy()

        if want_geometry:
            qn = Fn.normalize(QE, dim=-1)
            P = torch.stack(protos, dim=0)
            dist = (torch.cdist(qn.unsqueeze(0), P.unsqueeze(0)).squeeze(0) ** 2).detach().cpu().numpy()
            logits_u = model.classify(QE, protos_u)
            p_anx_u = Fn.softmax(logits_u, dim=-1)[:, pos_col].detach().float().cpu().numpy()
            pred_w = logits.detach().argmax(-1).cpu().numpy()
            pred_u = logits_u.detach().argmax(-1).cpu().numpy()
            allw = np.concatenate([wstat[c][0] for c in classes])
            alld = np.concatenate([wstat[c][1] for c in classes])
            hn, cvv, rat = [], [], []
            for c in classes:
                w = wstat[c][0]; kk = len(w)
                pp = np.clip(w, 1e-12, None); pp = pp / pp.sum()
                hn.append(float(-(pp * np.log(pp)).sum() / np.log(kk)) if kk > 1 else np.nan)
                cvv.append(float(w.std() / max(w.mean(), 1e-12)))
                rat.append(float(w.max() / max(w.min(), 1e-12)))
            corr = (float(np.corrcoef(allw, alld)[0, 1])
                    if allw.std() > 0 and alld.std() > 0 else np.nan)

        for j, (ridx, c_true) in enumerate(qry):
            r = store.records[ridx]
            row = {"episode": ep_i, "record_index": ridx,
                   "note_id": str(r["note_id"]), "patient_id": str(r["subject_id"]),
                   "label": int(c_true), "p_anxiety_note": float(p_anx[j])}
            if want_geometry:
                d_ctrl = float(dist[j, cls_to_col[0]]) if 0 in cls_to_col else np.nan
                d_case = float(dist[j, cls_to_col[1]]) if 1 in cls_to_col else np.nan
                row.update({
                    "d_proto_control": d_ctrl, "d_proto_case": d_case,
                    "margin_case_minus_control": d_ctrl - d_case,
                    "d_proto_own": d_case if c_true == 1 else d_ctrl,
                    "d_proto_other": d_ctrl if c_true == 1 else d_case,
                    "p_anxiety_note_uniform_w": float(p_anx_u[j]),
                    "decision_flips_without_weighting": bool(pred_w[j] != pred_u[j]),
                    "ep_H_norm": float(np.nanmean(hn)),
                    "ep_weight_cv": float(np.nanmean(cvv)),
                    "ep_weight_max_over_min": float(np.nanmean(rat)),
                    "ep_corr_weight_vs_days": corr,
                })
            rows.append(row)

    df = pd.DataFrame(rows)
    df["row_of"] = df["record_index"].map(row_of)
    return df, row_of, pooled.float().cpu().numpy(), proj.float().cpu().numpy()


def pool_to_patient(note_df):
    '''Patient-level pooling identical to tcwpn.evaluation.run_plan.'''
    g = note_df.groupby("patient_id")
    out = pd.DataFrame({
        "patient_id": g.size().index,
        "label": g["label"].first().values,
        "p_anxiety": g["p_anxiety_note"].mean().values,
        "n_episodes": g.size().values,
    })
    if (g["label"].nunique() > 1).any():
        raise RuntimeError("a patient appears with two labels; the cohort is inconsistent")
    return out.sort_values("patient_id").reset_index(drop=True)

print("helpers defined")

In [ ]:
# ---------------------------------------------------------------------------
# 0.3  Train / val / test performance for the run under investigation.
#      GPU. ~10-20 min depending on TRAIN_EPISODES.
# ---------------------------------------------------------------------------
TRAIN_EPISODES = 400            # cap: the train plan is 3000 random episodes

model, manifest = load_run(RUN_DIR)
THRESHOLD = float(manifest["locked_threshold"])
print(f"{RUN_NAME}  preset={manifest['config']['model'].get('preset')}  "
      f"K={manifest['k_shot']}  locked threshold={THRESHOLD:.5f} (validation-selected)")

split_rows, note_frames = [], {}
for split, cap in (("train", TRAIN_EPISODES), ("val", None), ("test", None)):
    pkl = PKL_DIR / f"{STEM}_{split}.pkl"
    plan_p = PLAN_DIR / f"{STEM}_{split}_k{K}.json"
    if not pkl.exists() or not plan_p.exists():
        print(f"  [{split}] missing pkl or plan -- skipped"); continue
    store = RecordStore.from_pkl(pkl, split_name=split)
    plan = EpisodePlan.load(plan_p)
    fp = plan.meta.get("store_fingerprint")
    if fp and fp != store_fingerprint(store):
        raise SystemExit(f"[{split}] plan/pkl fingerprint mismatch -- wrong Stage A dataset")

    ndf, *_ = score_plan_cached(model, store, plan, max_episodes=cap, desc=split)
    note_frames[split] = ndf
    pat = pool_to_patient(ndf)
    m = compute_metrics(pat["label"].values, pat["p_anxiety"].values,
                        threshold=THRESHOLD, n_bootstrap=1000)
    m.pop("_ece_table", None)
    m["split"] = split
    m["episodes_scored"] = int(ndf["episode"].nunique())
    m["coverage_note"] = "capped" if cap else "full plan"
    split_rows.append(m)

perf = pd.DataFrame(split_rows).set_index("split")
cols = ["n_patients", "prevalence", "auroc", "pr_auc", "f1_positive",
        "sensitivity", "specificity", "ppv", "npv", "brier", "ece",
        "episodes_scored", "coverage_note"]
perf = perf[[c for c in cols if c in perf.columns]]
print()
print(perf.round(4).to_string())
perf.to_csv(OUT / "phase6_split_performance.csv")

if {"train", "test"} <= set(perf.index):
    gap = float(perf.loc["train", "auroc"] - perf.loc["test", "auroc"])
    print(f"\ntrain-test AUROC gap: {gap:+.4f}")
    print("  gap > 0.10   -> overfitting")
    print("  gap ~ 0      -> underfitting OR an intrinsically overlapping problem")
    print("  sections 10-16 are what tell those two apart.")

## 1. Load frozen TC-WPN test predictions

`evaluate.py` writes `predictions_test.csv` with exactly:

```text
patient_id
label
p_anxiety
n_episodes
```

Two things are checked here rather than assumed:

1. **The recomputed note-level scores pool back to the committed patient-level file.** If the max
   absolute difference is not ~1e-6, the cached-embedding scorer in cell 0.2 is wrong and every
   number after this point is void. This is the guard that lets sections 10–15 use recomputed
   geometry and still describe the *same* frozen run.
2. **The threshold comes from `manifest.json`**, where `train.py` locked it on validation only.
   Nothing in this notebook selects a threshold on test.

In [ ]:
# ---------------------------------------------------------------------------
# 1.  Frozen test predictions + verification against the committed CSV.
# ---------------------------------------------------------------------------
pred_path = RUN_DIR / "predictions_test.csv"
if not pred_path.exists():
    print("predictions_test.csv is gitignored and was not in the imported run.")
    print("Regenerating it from best.pt with the LOCKED threshold ...")
    !python -m scripts.evaluate --run {RUN_DIR} --split test \
        --pkl-dir {PKL_DIR} --plan-dir {PLAN_DIR} --bootstrap 2000
if not pred_path.exists():
    raise SystemExit("could not obtain predictions_test.csv")

frozen = pd.read_csv(pred_path)
frozen["patient_id"] = frozen["patient_id"].astype(str)
print("committed predictions_test.csv")
print(frozen.head().to_string(index=False))
print(f"\n  columns   : {list(frozen.columns)}")
print(f"  patients  : {len(frozen):,}   case {int(frozen.label.sum()):,}   "
      f"control {int((frozen.label == 0).sum()):,}   "
      f"prevalence {frozen.label.mean():.4f}")

# ---- verification -----------------------------------------------------------
note_test = note_frames["test"]
recomputed = pool_to_patient(note_test)
chk = frozen.merge(recomputed, on="patient_id", suffixes=("_csv", "_recomputed"))
if len(chk) != len(frozen):
    raise SystemExit(f"only {len(chk)}/{len(frozen)} patients matched; wrong plan or wrong run")
max_abs = float((chk["p_anxiety_csv"] - chk["p_anxiety_recomputed"]).abs().max())
print(f"\n  patients matched                   : {len(chk):,}/{len(frozen):,}")
print(f"  max |p_csv - p_recomputed|         : {max_abs:.3e}")
print(f"  label agreement                    : {(chk.label_csv == chk.label_recomputed).all()}")
if max_abs > 1e-4:
    raise SystemExit(
        "recomputed probabilities do not reproduce predictions_test.csv. "
        "STOP -- sections 10-15 would be describing a different model.")
print("\n  VERIFIED: the recomputed note-level scores reproduce the frozen run.")

## 2. Identify TP / TN / FP / FN

Not accuracy. The four groups, separated, with the validation-locked threshold:

| group | actual | predicted |
|---|---|---|
| **TP** | anxiety | anxiety |
| **TN** | control | control |
| **FP** | control | anxiety |
| **FN** | anxiety | control |

**FP and FN are the objects of this entire investigation.**

Output: `error_analysis.csv`, with exactly the fields requested —
`patient_id`, `actual_label`, `predicted_probability`, `predicted_label`, `error_type`,
`n_episodes`.

One thing to read carefully before interpreting anything downstream: the locked threshold for this
run is 0.269 (selected to maximise F1 on validation), which buys sensitivity 0.922 at specificity
0.285. So **FN will be rare and FP will be common by construction**. That is a property of the
operating point, not of the clinical notes, and it must not be mistaken for a finding. A
threshold-free view (score distributions per class) is printed alongside for that reason.

In [ ]:
# ---------------------------------------------------------------------------
# 2.  TP / TN / FP / FN  ->  error_analysis.csv
# ---------------------------------------------------------------------------
err = frozen.rename(columns={"label": "actual_label",
                             "p_anxiety": "predicted_probability"}).copy()
err["predicted_label"] = (err["predicted_probability"] >= THRESHOLD).astype(int)

def label_error(r):
    if r.actual_label == 1 and r.predicted_label == 1: return "TP"
    if r.actual_label == 0 and r.predicted_label == 0: return "TN"
    if r.actual_label == 0 and r.predicted_label == 1: return "FP"
    return "FN"

err["error_type"] = err.apply(label_error, axis=1)
err["correct"] = err["error_type"].isin(["TP", "TN"])
err = err[["patient_id", "actual_label", "predicted_probability", "predicted_label",
           "error_type", "n_episodes", "correct"]]
err.to_csv(OUT / "error_analysis.csv", index=False)

counts = err["error_type"].value_counts().reindex(["TP", "TN", "FP", "FN"]).fillna(0).astype(int)
tp, tn, fp, fn = (int(counts[k]) for k in ("TP", "TN", "FP", "FN"))
print(f"threshold {THRESHOLD:.5f}  (locked on validation by train.py)\n")
print("                 predicted control   predicted anxiety")
print(f"  actual control        TN {tn:>6}          FP {fp:>6}")
print(f"  actual anxiety        FN {fn:>6}          TP {tp:>6}")
print(f"\n  sensitivity {tp/max(tp+fn,1):.4f}   specificity {tn/max(tn+fp,1):.4f}   "
      f"PPV {tp/max(tp+fp,1):.4f}   NPV {tn/max(tn+fn,1):.4f}")
print(f"  errors: {fp+fn:,} of {len(err):,} patients ({(fp+fn)/len(err):.1%})")

print("\nWorked example of the table the supervisor asked for "
      "(2 correct + 2 errors, actual patients):")
show = pd.concat([err[err.error_type == t].head(1) for t in ("TP", "TN", "FN", "FP")])
print(show.to_string(index=False))

print("\nThreshold-free view -- predicted probability by true class:")
print(err.groupby("actual_label")["predicted_probability"]
        .describe()[["count", "mean", "std", "25%", "50%", "75%"]].round(4).to_string())
print("\nCAUTION: at a threshold of 0.269 the FN group is small by construction. "
      "Read FN patterns as suggestive, and weight FP evidence accordingly.")

print(f"\nwrote {OUT/'error_analysis.csv'}")

## 3. Link predictions to patient notes

```text
patient_id  ->  note IDs  ->  clinical note  ->  note metadata
```

The frozen test plan says exactly which record indices were queried in which episode, so the
note-level link is **deterministic and re-derivable** — no guessing about which notes a patient
contributed.

Three text views are built, and they are not interchangeable:

| view | what it is | why it matters |
|---|---|---|
| `text_full` | the normalised note in `cohort_psych_mimic4idx.csv` | what a clinician would read |
| `text_model_saw` | the pkl token ids decoded back with the Bio_ClinicalBERT tokenizer | `tokenize_cohort.py` runs with `max_chunks=1`, so the model sees **only the first 512 tokens** |
| `truncated` | whether the two differ | a note whose anxiety evidence sits past token 512 is invisible to the model — that is a pipeline/data root cause, not a model one, and it is measurable |

Decoding the pkl is the same technique `run_shallow_baselines.py` uses, and it guarantees the
characteristics in sections 5–9 are measured on the text the model actually consumed.

In [ ]:
# ---------------------------------------------------------------------------
# 3.  patient -> episode -> note -> text -> metadata
# ---------------------------------------------------------------------------
test_store = RecordStore.from_pkl(PKL_DIR / f"{STEM}_test.pkl", split_name="test")
cohort = pd.read_csv(COHORT_CSV, low_memory=False)
cohort["note_id"] = cohort["note_id"].astype(str)
cohort["subject_id"] = cohort["subject_id"].astype(str)
print(f"cohort rows {len(cohort):,} | columns present:")
print("   " + ", ".join(cohort.columns))

cohort_test = cohort[cohort["split"] == "test"].copy()
print(f"\ntest-split cohort rows: {len(cohort_test):,} "
      f"({cohort_test.subject_id.nunique():,} patients)")

# ---- decode exactly what the model read ------------------------------------
from transformers import AutoTokenizer
ENCODER = manifest["config"]["model"].get("encoder_name", "emilyalsentzer/Bio_ClinicalBERT")
tok = AutoTokenizer.from_pretrained(ENCODER)

queried = sorted(set(note_test["record_index"].astype(int)))
seen = {}
for ix in tqdm(queried, desc="decoding what the model saw", leave=False):
    r = test_store.records[ix]
    ids = [t for chunk in r["input_ids"] for t in chunk]
    ntok = int(sum(sum(c) for c in r["attention_mask"]))
    seen[ix] = (tok.decode(ids, skip_special_tokens=True), ntok)

meta = pd.DataFrame({
    "record_index": queried,
    "note_id": [str(test_store.records[i]["note_id"]) for i in queried],
    "patient_id": [str(test_store.records[i]["subject_id"]) for i in queried],
    "text_model_saw": [seen[i][0] for i in queried],
    "n_tokens_model_saw": [seen[i][1] for i in queried],
    "days_in_pkl": [float(test_store.records[i].get("days_before_patient_last_note", 0.0))
                    for i in queried],
})

joined = meta.merge(
    cohort_test.rename(columns={"subject_id": "patient_id"}),
    on=["note_id", "patient_id"], how="left")

unmatched = int(joined["text"].isna().sum())
print(f"\nnotes queried in the test plan : {len(meta):,}")
print(f"joined to cohort text          : {len(joined) - unmatched:,}")
print(f"UNMATCHED                      : {unmatched:,}")
if unmatched:
    raise SystemExit("some queried notes have no cohort row -- the cohort CSV does not "
                     "match the pkl the plan was built from.")

joined = joined.rename(columns={"text": "text_full"})
joined["text_full"] = joined["text_full"].fillna("")
joined["truncated_by_512"] = joined["text_full"].str.len() > joined["text_model_saw"].str.len() + 5

# temporal field sanity: after apply_index_time the model reads days_before_index
if "days_before_index" in joined.columns:
    dd = (joined["days_in_pkl"] - joined["days_before_index"]).abs()
    print(f"\nmax |days_in_pkl - days_before_index| = {dd.max():.4f} "
          f"(0 confirms w^T is driven by days_before_index, as apply_index_time.py states)")

print(f"\ntruncated at 512 tokens: {joined.truncated_by_512.mean():.1%} of queried notes")
print(f"tokens the model saw   : median {joined.n_tokens_model_saw.median():.0f}, "
      f"max {joined.n_tokens_model_saw.max():.0f}")

## 4. Build note-level error table

Two units of analysis, kept separate on purpose:

* **Patient level** — the unit the paper reports. A patient is TP/TN/FP/FN once, from the pooled
  probability across the ~3.6 episodes they were queried in, against the locked threshold.
* **Note level** — the unit the supervisor's question is about ("*which clinical notes*"). One row
  per queried note, carrying that note's own probability, its patient's error type, and the note's
  metadata.

A patient can contribute several notes and a note can appear in several episodes, so note-level
rows are **not independent**. Every test in sections 5–9 is therefore run twice: once over notes,
and once over per-patient means (one row per patient, the conservative version). Where the two
disagree, the patient-level result is the one to believe. This is the same reason
`tcwpn.metrics` bootstraps patients rather than episodes.

In [ ]:
# ---------------------------------------------------------------------------
# 4.  note-level error table
# ---------------------------------------------------------------------------
note_scores = (note_test.groupby(["record_index", "note_id", "patient_id", "label"],
                                 as_index=False)
               .agg(p_anxiety_note=("p_anxiety_note", "mean"),
                    n_episodes_note=("episode", "nunique")))

notes = (note_scores
         .merge(joined.drop(columns=["label"], errors="ignore"),
                on=["record_index", "note_id", "patient_id"], how="left")
         .merge(err[["patient_id", "error_type", "correct", "predicted_probability",
                     "predicted_label", "actual_label"]],
                on="patient_id", how="left"))

assert notes["error_type"].notna().all(), "a queried note has no patient-level verdict"
notes["note_pred_label"] = (notes["p_anxiety_note"] >= THRESHOLD).astype(int)
notes["note_correct"] = notes["note_pred_label"] == notes["label"]

print(f"note-level rows: {len(notes):,}   patients: {notes.patient_id.nunique():,}")
print(f"notes per patient: mean {len(notes)/notes.patient_id.nunique():.2f}, "
      f"max {notes.groupby('patient_id').size().max()}")
print()
print("notes by PATIENT-level verdict:")
print(notes["error_type"].value_counts().reindex(["TP", "TN", "FP", "FN"]).fillna(0)
      .astype(int).to_string())
print()
print("agreement between the note's own verdict and its patient's verdict: "
      f"{(notes.note_correct == notes.correct).mean():.1%}")

KEEP = ["record_index", "note_id", "patient_id", "label", "actual_label",
        "p_anxiety_note", "n_episodes_note", "predicted_probability", "predicted_label",
        "error_type", "correct", "note_pred_label", "note_correct",
        "n_tokens_model_saw", "truncated_by_512", "days_in_pkl"]
for c in ["note_source", "charttime", "hadm_id", "gender", "age", "arm",
          "days_before_index", "days_before_patient_last_note",
          "days_before_last_note_preindex", "days_before_patient_last_note_raw",
          "days_since_patient_first_note", "note_index_within_patient",
          "n_notes_patient", "is_patient_last_note", "anx_coded_this_adm"]:
    if c in notes.columns:
        KEEP.append(c)
notes[KEEP].to_csv(OUT / "note_error_table.csv", index=False)
print(f"\nwrote {OUT/'note_error_table.csv'}  ({len(KEEP)} columns, text excluded)")

In [ ]:
# ---------------------------------------------------------------------------
# 4b. Statistical helpers used by sections 5-9.
#     Cliff's delta for continuous, risk difference + Fisher for binary,
#     Holm-Bonferroni across the whole family of tests in section 9.
# ---------------------------------------------------------------------------
from scipy import stats

def cliffs_delta(a, b):
    '''delta = 2U/(n1*n2) - 1. +1 = a always larger. |d|: .11 small, .28 medium, .43 large.'''
    a = np.asarray(a, float); a = a[~np.isnan(a)]
    b = np.asarray(b, float); b = b[~np.isnan(b)]
    if len(a) < 2 or len(b) < 2:
        return np.nan, np.nan
    u, p = stats.mannwhitneyu(a, b, alternative="two-sided")
    return float(2.0 * u / (len(a) * len(b)) - 1.0), float(p)

def continuous_test(df, col, group_col="correct"):
    a = df.loc[~df[group_col], col]     # incorrect
    b = df.loc[df[group_col], col]      # correct
    d, p = cliffs_delta(a, b)
    return {"characteristic": col, "kind": "continuous",
            "n_incorrect": int(a.notna().sum()), "n_correct": int(b.notna().sum()),
            "incorrect_median": float(np.nanmedian(a)) if len(a) else np.nan,
            "correct_median": float(np.nanmedian(b)) if len(b) else np.nan,
            "incorrect_mean": float(np.nanmean(a)) if len(a) else np.nan,
            "correct_mean": float(np.nanmean(b)) if len(b) else np.nan,
            "effect": d, "effect_name": "cliffs_delta", "p_raw": p}

def binary_test(df, col, group_col="correct"):
    a = df.loc[~df[group_col], col].astype(bool)
    b = df.loc[df[group_col], col].astype(bool)
    if len(a) < 2 or len(b) < 2:
        return None
    table = [[int(a.sum()), int((~a).sum())], [int(b.sum()), int((~b).sum())]]
    try:
        _, p = stats.fisher_exact(table)
    except Exception:
        _, p, _, _ = stats.chi2_contingency(np.array(table) + 0.5)
    return {"characteristic": col, "kind": "binary",
            "n_incorrect": int(len(a)), "n_correct": int(len(b)),
            "incorrect_median": float(a.mean()), "correct_median": float(b.mean()),
            "incorrect_mean": float(a.mean()), "correct_mean": float(b.mean()),
            "effect": float(a.mean() - b.mean()), "effect_name": "risk_difference",
            "p_raw": float(p)}

def holm(pvals):
    '''Holm-Bonferroni adjusted p-values, order preserved.'''
    p = np.asarray(pvals, float)
    ok = ~np.isnan(p)
    out = np.full_like(p, np.nan)
    idx = np.argsort(p[ok]); m = ok.sum()
    adj = np.empty(m); run = 0.0
    ps = p[ok][idx]
    for i in range(m):
        run = max(run, (m - i) * ps[i])
        adj[i] = min(run, 1.0)
    tmp = np.empty(m); tmp[idx] = adj
    out[ok] = tmp
    return out

def patient_view(note_df, cols):
    '''One row per patient: numeric cols averaged, flags -> any().'''
    agg = {}
    for c in cols:
        if c not in note_df.columns:
            continue
        agg[c] = "max" if pd.api.types.is_bool_dtype(note_df[c]) else "mean"
    g = note_df.groupby("patient_id").agg({**agg, "correct": "first",
                                           "error_type": "first", "label": "first"})
    return g.reset_index()

TESTS = []   # every section 5-8 test appends here; section 9 corrects the family
print("statistical helpers defined")

## 5. Analyze note length

*"Maybe errors are concentrated in very short notes. But we must test this, not assume it."*

Measured three ways, on both text views:

* `n_chars_full`, `n_words_full` — the full normalised note
* `n_tokens_model_saw` — Bio_ClinicalBERT wordpiece tokens actually consumed (`max_chunks=1`
  caps this at 512, so it saturates; that saturation is itself informative)
* `truncated_by_512` — whether evidence could have been cut off

One thing to watch: `build_clean_cohort.py` already drops notes under 250 characters
(`MIN_NOTE_CHARS`), identically for both classes, so the very short tail does not exist in this
cohort. If a length effect appears it will be within the surviving range.

In [ ]:
# ---------------------------------------------------------------------------
# 5.  note length
# ---------------------------------------------------------------------------
notes["n_chars_full"] = notes["text_full"].str.len()
notes["n_words_full"] = notes["text_full"].str.split().str.len().fillna(0)
notes["frac_note_seen"] = (notes["text_model_saw"].str.len()
                           / notes["n_chars_full"].clip(lower=1)).clip(upper=1.0)

LEN_COLS = ["n_chars_full", "n_words_full", "n_tokens_model_saw", "frac_note_seen"]

print("by patient-level verdict (median):")
print(notes.groupby("error_type")[LEN_COLS].median()
      .reindex(["TP", "TN", "FP", "FN"]).round(2).to_string())
print("\nminimum note length in the cohort (build_clean_cohort MIN_NOTE_CHARS = 250): "
      f"{int(notes.n_chars_full.min())} chars")

print("\ncorrect vs incorrect -- NOTE level:")
rows = [continuous_test(notes, c) for c in LEN_COLS]
rows.append(binary_test(notes, "truncated_by_512"))
tbl = pd.DataFrame([r for r in rows if r])
print(tbl[["characteristic", "incorrect_median", "correct_median",
           "effect", "p_raw"]].round(4).to_string(index=False))
for r in rows:
    if r: TESTS.append({**r, "section": "5_length", "unit": "note"})

pv = patient_view(notes, LEN_COLS + ["truncated_by_512"])
print("\ncorrect vs incorrect -- PATIENT level (conservative):")
prows = [continuous_test(pv, c) for c in LEN_COLS] + [continuous_test(pv, "truncated_by_512")]
ptbl = pd.DataFrame(prows)
print(ptbl[["characteristic", "incorrect_median", "correct_median",
            "effect", "p_raw"]].round(4).to_string(index=False))
for r in prows:
    TESTS.append({**r, "section": "5_length", "unit": "patient"})

print("\nReading: |cliffs_delta| < 0.11 is negligible however small p becomes -- with "
      f"{len(notes):,} notes, trivial differences reach significance.")

## 6. Analyze note type

*"If the available metadata permits it, compare discharge summaries, progress notes, physician
notes, nursing notes, psychiatric notes..."* and *"Don't invent categories that aren't present in
your source data."*

Those two instructions collide here, and the second one wins. `build_clean_cohort.py` streams
MIMIC-IV notes from `discharge.csv` only and hard-codes `note_source = "discharge"`. The
multi-category comparison is only available on the MIMIC-III arm, which is reserved for
cross-dataset transfer and is not part of this benchmark.

So this cell **measures** how many note types exist rather than assuming one, and reports the
outcome honestly. If `note_source` has a single value, the correct thing to write in the paper is
that the note-type comparison was not available in this cohort — not to substitute a text-derived
pseudo-category, which would reintroduce exactly the text-derived construct
`tests/test_no_text_derived_filtering.py` exists to prevent.

What *is* legitimately available is **structural** metadata: which section headers a discharge
summary contains. That is measured in section 8, where it belongs.

In [ ]:
# ---------------------------------------------------------------------------
# 6.  note type -- report what exists, do not invent categories
# ---------------------------------------------------------------------------
if "note_source" not in notes.columns:
    print("note_source is absent from the cohort CSV; note-type analysis not available.")
else:
    vc = notes["note_source"].value_counts()
    print("note_source values present in the queried test notes:")
    print(vc.to_string())
    if vc.nunique() <= 1 or len(vc) == 1:
        print(f"\n  Only ONE note type is present ('{vc.index[0]}').")
        print("  build_clean_cohort.py reads MIMIC-IV discharge.csv and sets")
        print("  note_source = 'discharge' for every row, so a note-type comparison")
        print("  is NOT AVAILABLE in this cohort.")
        print("\n  For the paper: state this as a stated limitation. Do not manufacture")
        print("  note categories from the text -- that is the text-derived construct")
        print("  the clean rebuild removed.")
    else:
        ct = pd.crosstab(notes["note_source"], notes["error_type"], normalize="index")
        print("\nerror-type composition by note type:")
        print(ct.round(4).to_string())
        chi = stats.chi2_contingency(pd.crosstab(notes["note_source"], notes["correct"]))
        print(f"\nchi2 note_source x correct: chi2={chi[0]:.2f}  p={chi[1]:.4g}")
        TESTS.append({"characteristic": "note_source", "kind": "categorical",
                      "section": "6_note_type", "unit": "note",
                      "n_incorrect": int((~notes.correct).sum()),
                      "n_correct": int(notes.correct.sum()),
                      "incorrect_median": np.nan, "correct_median": np.nan,
                      "incorrect_mean": np.nan, "correct_mean": np.nan,
                      "effect": np.nan, "effect_name": "cramers_v_not_computed",
                      "p_raw": float(chi[1])})

for c in ["gender", "arm"]:
    if c in notes.columns and notes[c].nunique() > 1:
        print(f"\n{c} x error_type (row-normalised) -- available metadata, reported for completeness:")
        print(pd.crosstab(notes[c], notes["error_type"], normalize="index").round(3).to_string())

## 7. Analyze temporal distance

*"This is particularly important because TC-WPN claims a temporal mechanism."*

The quantity is `days_before_index`. Under `apply_index_time.py --policy at_or_before` it is the
gap between a note's `charttime` and the discharge of the index admission, and it is written into
`days_before_patient_last_note`, which is the column `tokenize_cohort.py` stores and
`TemporalRecencyWeight` consumes as `dt` in

```text
w^T = exp(-lambda * dt / 365)
```

with a learned `lambda` that finished at **0.4378** for this run (from `manifest.json`). At that
lambda a note one year before index keeps weight e^-0.4378 = 0.645 of a note at index.

The supervisor's two branches:

* errors clustered in **older** notes → directly relevant to the temporal mechanism
* errors clustered in **recent** notes → something else is going on

A third possibility worth naming in advance: **no temporal difference at all**, which would say the
temporal mechanism has nothing to grip on in this cohort and would explain `temporal_aux` ≈
`aux_only` without invoking any bug.

In [ ]:
# ---------------------------------------------------------------------------
# 7.  temporal distance
# ---------------------------------------------------------------------------
lam = None
for h in reversed(manifest.get("history", [])):
    if h.get("lambda_decay") is not None:
        lam = float(h["lambda_decay"]); break
print(f"learned lambda at the end of training: {lam}")
if lam:
    for yrs in (0.5, 1, 2, 5):
        print(f"   a note {yrs:>3} year(s) before index keeps w^T = "
              f"{np.exp(-lam * yrs):.4f} of a note at index")

TCOLS = [c for c in ["days_in_pkl", "days_before_index", "days_since_patient_first_note",
                     "note_index_within_patient", "n_notes_patient"]
         if c in notes.columns]

print("\nby patient-level verdict (median):")
print(notes.groupby("error_type")[TCOLS].median()
      .reindex(["TP", "TN", "FP", "FN"]).round(2).to_string())

print("\ndistribution of days_before_index (all queried test notes):")
print(notes["days_in_pkl"].describe(percentiles=[.25, .5, .75, .9, .99]).round(2).to_string())
at_index = float((notes["days_in_pkl"] <= 0.5).mean())
print(f"  notes AT the index discharge (dt <= 0.5 d): {at_index:.1%}")
if at_index > 0.8:
    print("  -> the temporal weight has almost no spread to act on in this cohort.")
    print("     That alone would explain temporal_aux ~ aux_only without any bug.")

print("\ncorrect vs incorrect -- NOTE level:")
rows = [continuous_test(notes, c) for c in TCOLS]
print(pd.DataFrame(rows)[["characteristic", "incorrect_median", "correct_median",
                          "effect", "p_raw"]].round(4).to_string(index=False))
for r in rows:
    TESTS.append({**r, "section": "7_temporal", "unit": "note"})

pv7 = patient_view(notes, TCOLS)
prows = [continuous_test(pv7, c) for c in TCOLS]
print("\ncorrect vs incorrect -- PATIENT level:")
print(pd.DataFrame(prows)[["characteristic", "incorrect_median", "correct_median",
                           "effect", "p_raw"]].round(4).to_string(index=False))
for r in prows:
    TESTS.append({**r, "section": "7_temporal", "unit": "patient"})

if "is_patient_last_note" in notes.columns:
    b = binary_test(notes, "is_patient_last_note")
    if b:
        print(f"\nis_patient_last_note: {b['incorrect_mean']:.3f} of incorrect vs "
              f"{b['correct_mean']:.3f} of correct  (p={b['p_raw']:.4g})")
        TESTS.append({**b, "section": "7_temporal", "unit": "note"})

## 8. Analyze clinical-text characteristics

*"We should measure whether errors contain: direct anxiety terminology, indirect symptom
terminology, psychiatric terminology, medication references, negation, family/social history,
unrelated clinical terminology."*

And the constraint that goes with it: *"don't use these terms to 'fix' the labels. We're
investigating model behavior, not rebuilding the label."* Nothing here touches a label, a split, a
cohort membership decision, or a training weight. These are **measurement instruments applied after
the fact to a frozen prediction file**.

Provenance of every vocabulary, stated so a reviewer can check it:

| instrument | source |
|---|---|
| direct anxiety terms | `scripts/tokenize_cohort.py :: ANXIETY_TERMS` — imported, not retyped |
| anxiolytic / antidepressant terms | `scripts/tokenize_cohort.py :: MED_TERMS` — imported |
| other psychiatric terms | `scripts/tokenize_cohort.py :: PSYCH_TERMS` — imported |
| indirect symptom terms | **new, declared in this notebook** — not derived from the cohort, not used anywhere in training |
| negation cues | **new, declared in this notebook** — crude pre-term window matcher, limitations stated below |
| section headers | discovered empirically from the notes, then counted |

The negation detector is a window matcher (a cue within 6 tokens before an anxiety term), not
NegEx and not a parser. It will miss post-posed negation and it will over-fire on
`"no acute distress ... anxiety noted"`. It is reported as a screening signal only, and the
per-note hit rate is printed so the crudeness is visible rather than hidden.

Measured on **both** text views, because the difference between them is itself a candidate root
cause: `anxiety_in_full_but_not_in_window` counts notes where the model was structurally unable to
see the evidence.

The three highest-value derived flags, which map directly onto the supervisor's example table:

* `anx_only_in_family_or_social` — "anxiety-related terminology but no anxiety diagnosis"
* `anx_all_negated` — terminology present but denied
* `anxiety_in_full_but_not_in_window` — evidence past token 512

Context for reading these: `audit_cohort.py` already established that anxiety terms appear at very
different rates per class, and the blinded runs already showed AUROC 0.7379 → 0.6284 when anxiety
terms are deleted. Roughly **46% of the above-chance margin is lexical**. So lexical flags will be
strong predictors of correctness — that is expected and already known. What is new here is
*which errors* they explain.

In [ ]:
# ---------------------------------------------------------------------------
# 8a. Vocabularies. Repo vocabularies are IMPORTED so they cannot drift.
# ---------------------------------------------------------------------------
import importlib.util, re

spec = importlib.util.spec_from_file_location(
    "tok_cohort", "/kaggle/working/tcwpn_test/scripts/tokenize_cohort.py")
tok_cohort = importlib.util.module_from_spec(spec)
spec.loader.exec_module(tok_cohort)
ANXIETY_TERMS = tok_cohort.ANXIETY_TERMS
MED_TERMS     = tok_cohort.MED_TERMS
PSYCH_TERMS   = tok_cohort.PSYCH_TERMS
print(f"imported from tokenize_cohort.py: {len(ANXIETY_TERMS)} anxiety, "
      f"{len(MED_TERMS)} medication, {len(PSYCH_TERMS)} psychiatric terms")

# --- NEW instruments, declared here, used for measurement only -------------
INDIRECT_SYMPTOM_TERMS = [
    "worry", "worried", "worrying", "restless", "restlessness", "on edge", "edgy",
    "tense", "tension", "insomnia", "difficulty sleeping", "trouble sleeping",
    "palpitations", "racing heart", "tachycardia", "shortness of breath",
    "chest tightness", "hyperventilation", "dizziness", "lightheaded", "tremor",
    "tremulous", "shaky", "diaphoresis", "sweating", "nausea", "hypervigilance",
    "fear", "fearful", "apprehensive", "apprehension", "irritable", "irritability",
    "difficulty concentrating", "poor concentration", "agitation", "agitated",
    "stress", "stressed", "overwhelmed", "distress", "distressed",
]
NEGATION_CUES = [
    "no", "not", "never", "denies", "denied", "denying", "without", "negative for",
    "free of", "absent", "unremarkable for", "rule out", "r/o", "ruled out", "unlikely",
]
UNRELATED_CLINICAL_TERMS = [   # a "is this note about something else entirely" probe
    "sepsis", "pneumonia", "myocardial infarction", "stemi", "nstemi", "fracture",
    "appendicitis", "cholecystitis", "pancreatitis", "gi bleed", "hemorrhage",
    "stroke", "seizure", "transplant", "dialysis", "copd", "cellulitis",
    "diabetic ketoacidosis", "dka", "trauma", "obstruction", "carcinoma", "chemotherapy",
]

def pat(terms):
    ordered = sorted(set(terms), key=len, reverse=True)
    return re.compile(r"\b(?:" + "|".join(re.escape(t) for t in ordered) + r")\b",
                      flags=re.IGNORECASE)

P_ANX, P_MED, P_PSY = pat(ANXIETY_TERMS), pat(MED_TERMS), pat(PSYCH_TERMS)
P_IND, P_UNR = pat(INDIRECT_SYMPTOM_TERMS), pat(UNRELATED_CLINICAL_TERMS)
P_NEG = pat(NEGATION_CUES)

# --- section headers: discovered from the data, not assumed ----------------
P_HEADER = re.compile(r"(?m)^[ \t]*([A-Z][A-Za-z][A-Za-z '/&\-]{2,45}):")
from collections import Counter
hdr_counter = Counter()
for t in notes["text_full"].head(3000):
    hdr_counter.update({h.strip().lower() for h in P_HEADER.findall(t or "")})
print(f"\nmost common section headers found in the first 3,000 test notes "
      f"(discovered, not assumed):")
for h, n in hdr_counter.most_common(20):
    print(f"   {n:>6}  {h}")

FAMSOC = [h for h in ("family history", "social history") if h in hdr_counter]
print(f"\nfamily/social history headers usable: {FAMSOC if FAMSOC else 'NONE FOUND'}")

In [ ]:
# ---------------------------------------------------------------------------
# 8b. Per-note measurement.
# ---------------------------------------------------------------------------
def sections_of(text):
    '''Split a note into {header_lower: body} using the discovered header pattern.'''
    if not isinstance(text, str) or not text:
        return {}
    marks = [(m.start(), m.end(), m.group(1).strip().lower())
             for m in P_HEADER.finditer(text)]
    out = {}
    for i, (s, e, name) in enumerate(marks):
        stop = marks[i + 1][0] if i + 1 < len(marks) else len(text)
        out.setdefault(name, "")
        out[name] += " " + text[e:stop]
    return out

def negated_hits(text, p_term, window=6):
    '''Fraction of term hits with a negation cue in the <window> tokens before it.'''
    if not isinstance(text, str) or not text:
        return 0, 0
    toks = text.split()
    low = [t.lower().strip(".,;:()") for t in toks]
    joined_idx, total, neg = 0, 0, 0
    for m in p_term.finditer(text):
        total += 1
        upto = len(text[:m.start()].split())
        lo = max(0, upto - window)
        if any(c in low[lo:upto] for c in ("no", "not", "never", "denies", "denied",
                                           "denying", "without", "absent", "unlikely")) \
           or re.search(r"(negative for|free of|rule[d]? out|r/o)\s*$",
                        text[max(0, m.start() - 40):m.start()], flags=re.I):
            neg += 1
    return neg, total

feat = []
for t_full, t_saw in tqdm(zip(notes["text_full"], notes["text_model_saw"]),
                          total=len(notes), desc="measuring text", leave=False):
    t_full = t_full or ""; t_saw = t_saw or ""
    secs = sections_of(t_full)
    famsoc_text = " ".join(secs.get(h, "") for h in FAMSOC)
    body_text = " ".join(v for k, v in secs.items() if k not in FAMSOC) or t_full
    n_anx_full = len(P_ANX.findall(t_full))
    n_anx_saw  = len(P_ANX.findall(t_saw))
    neg_hits, tot_hits = negated_hits(t_saw, P_ANX)
    feat.append({
        "anx_terms_seen": n_anx_saw,
        "anx_terms_full": n_anx_full,
        "has_anx_term_seen": n_anx_saw > 0,
        "has_anx_term_full": n_anx_full > 0,
        "anxiety_in_full_but_not_in_window": (n_anx_full > 0) and (n_anx_saw == 0),
        "med_terms_seen": len(P_MED.findall(t_saw)),
        "has_med_term_seen": len(P_MED.findall(t_saw)) > 0,
        "psych_terms_seen": len(P_PSY.findall(t_saw)),
        "has_psych_term_seen": len(P_PSY.findall(t_saw)) > 0,
        "indirect_terms_seen": len(P_IND.findall(t_saw)),
        "has_indirect_term_seen": len(P_IND.findall(t_saw)) > 0,
        "unrelated_terms_seen": len(P_UNR.findall(t_saw)),
        "n_sections": len(secs),
        "has_famsoc_section": bool(famsoc_text.strip()),
        "anx_in_famsoc": len(P_ANX.findall(famsoc_text)) > 0,
        "anx_only_in_family_or_social": (len(P_ANX.findall(famsoc_text)) > 0
                                         and len(P_ANX.findall(body_text)) == 0),
        "anx_negation_rate_seen": (neg_hits / tot_hits) if tot_hits else np.nan,
        "anx_all_negated": bool(tot_hits > 0 and neg_hits == tot_hits),
        "indirect_only_no_direct": (len(P_IND.findall(t_saw)) > 0 and n_anx_saw == 0),
    })
notes = pd.concat([notes.reset_index(drop=True), pd.DataFrame(feat)], axis=1)

TEXT_BIN = ["has_anx_term_seen", "has_med_term_seen", "has_psych_term_seen",
            "has_indirect_term_seen", "indirect_only_no_direct", "anx_all_negated",
            "anx_in_famsoc", "anx_only_in_family_or_social", "has_famsoc_section",
            "anxiety_in_full_but_not_in_window"]
TEXT_CNT = ["anx_terms_seen", "med_terms_seen", "psych_terms_seen",
            "indirect_terms_seen", "unrelated_terms_seen", "n_sections"]
if "anx_coded_this_adm" in notes.columns:
    notes["anx_coded_this_adm"] = notes["anx_coded_this_adm"].astype(bool)
    TEXT_BIN.append("anx_coded_this_adm")

print("\nprevalence of each measured characteristic, by patient-level verdict:")
print(notes.groupby("error_type")[TEXT_BIN].mean()
      .reindex(["TP", "TN", "FP", "FN"]).round(3).to_string())
print("\nmean counts by verdict:")
print(notes.groupby("error_type")[TEXT_CNT].mean()
      .reindex(["TP", "TN", "FP", "FN"]).round(2).to_string())

print("\nsanity check on the lexical shortcut already known from audit_cohort.py:")
print(notes.groupby("label")["has_anx_term_seen"].mean().round(3).to_string())
print("  (label 1 = anxiety case. A large gap here is the shortcut the blinded")
print("   runs quantified: AUROC 0.7379 unblinded -> 0.6284 anxiety-blinded.)")

rows = ([binary_test(notes, c) for c in TEXT_BIN]
        + [continuous_test(notes, c) for c in TEXT_CNT]
        + [continuous_test(notes, "anx_negation_rate_seen")])
rows = [r for r in rows if r]
print("\ncorrect vs incorrect -- NOTE level:")
print(pd.DataFrame(rows)[["characteristic", "incorrect_mean", "correct_mean",
                          "effect", "p_raw"]].round(4).to_string(index=False))
for r in rows:
    TESTS.append({**r, "section": "8_text", "unit": "note"})

pv8 = patient_view(notes, TEXT_BIN + TEXT_CNT)
prows = [continuous_test(pv8, c) for c in TEXT_BIN + TEXT_CNT if c in pv8.columns]
print("\ncorrect vs incorrect -- PATIENT level:")
print(pd.DataFrame(prows)[["characteristic", "incorrect_mean", "correct_mean",
                           "effect", "p_raw"]].round(4).to_string(index=False))
for r in prows:
    TESTS.append({**r, "section": "8_text", "unit": "patient"})

notes.drop(columns=["text_full", "text_model_saw"]).to_csv(
    OUT / "note_error_table_with_characteristics.csv", index=False)
print(f"\nwrote {OUT/'note_error_table_with_characteristics.csv'}")

## 9. Compare correct vs incorrect groups

Everything measured in sections 5–8 is pulled into one family and corrected together.

Why correction is not optional here: roughly 60 tests are run over ~8 000 note rows. At n = 8 000 a
Cliff's δ of 0.05 — a difference no one could act on — reaches p < 0.001. Reporting raw p-values
from this table would manufacture findings.

The rule used, fixed before looking:

* **Holm-Bonferroni** across the whole family
* a characteristic is a **candidate stratum** only if adjusted p < 0.05 **and** |effect| clears the
  practical floor: |Cliff's δ| ≥ 0.33 (medium) for continuous, |risk difference| ≥ 0.15 for binary
* the **patient-level** result is the one that counts; note-level rows are not independent

Also reported: FP-specific and FN-specific contrasts (each error class against its own correctly
classified counterpart — FP vs TN, FN vs TP), because a characteristic that separates FP from TN is
a *false-positive* driver and a characteristic that separates FN from TP is a *false-negative*
driver, and pooling them can cancel both out.

In [ ]:
# ---------------------------------------------------------------------------
# 9.  the family, corrected together
# ---------------------------------------------------------------------------
fam = pd.DataFrame(TESTS)
fam["p_holm"] = holm(fam["p_raw"].values)
fam["practical_floor"] = np.where(fam["effect_name"] == "risk_difference", 0.15, 0.33)
fam["candidate"] = (fam["p_holm"] < 0.05) & (fam["effect"].abs() >= fam["practical_floor"])
fam = fam.sort_values(["unit", "candidate", "effect"], ascending=[True, False, False])
fam.to_csv(OUT / "phase6_characteristic_tests.csv", index=False)

print(f"tests in the family: {len(fam)}   "
      f"raw p<0.05: {(fam.p_raw < 0.05).sum()}   "
      f"Holm p<0.05: {(fam.p_holm < 0.05).sum()}   "
      f"AND effect above the practical floor: {int(fam.candidate.sum())}")

print("\nTOP 15 BY ABSOLUTE EFFECT SIZE (patient level, the conservative unit):")
top = (fam[fam.unit == "patient"].reindex(fam[fam.unit == "patient"].effect.abs()
       .sort_values(ascending=False).index).head(15))
print(top[["section", "characteristic", "incorrect_mean", "correct_mean", "effect",
           "effect_name", "p_raw", "p_holm", "candidate"]].round(4).to_string(index=False))

CANDIDATES = sorted(set(fam.loc[fam.candidate & (fam.unit == "patient"), "characteristic"]))
print(f"\nCANDIDATE STRATA (patient level, both criteria met): "
      f"{CANDIDATES if CANDIDATES else 'NONE'}")
if not CANDIDATES:
    print("  No note characteristic separates correct from incorrect patients at a")
    print("  practically meaningful effect size. That is a real result: it argues")
    print("  AGAINST a data root cause and pushes the diagnosis toward sections 11-15.")

# ---- error-class-specific contrasts ----------------------------------------
ALLCH = [c for c in (LEN_COLS + TCOLS + TEXT_BIN + TEXT_CNT) if c in notes.columns]
pv_all = patient_view(notes, ALLCH)

def contrast(err_type, ref_type, name):
    sub = pv_all[pv_all.error_type.isin([err_type, ref_type])].copy()
    sub["is_err"] = sub.error_type == err_type
    out = []
    for c in ALLCH:
        a = sub.loc[sub.is_err, c]; b = sub.loc[~sub.is_err, c]
        d, p = cliffs_delta(a, b)
        out.append({"contrast": name, "characteristic": c,
                    f"{err_type}_mean": float(np.nanmean(a)) if len(a) else np.nan,
                    f"{ref_type}_mean": float(np.nanmean(b)) if len(b) else np.nan,
                    "cliffs_delta": d, "p_raw": p, "n_err": int(a.notna().sum()),
                    "n_ref": int(b.notna().sum())})
    o = pd.DataFrame(out)
    o["p_holm"] = holm(o["p_raw"].values)
    return o.reindex(o.cliffs_delta.abs().sort_values(ascending=False).index)

fp_vs_tn = contrast("FP", "TN", "FP_vs_TN")
fn_vs_tp = contrast("FN", "TP", "FN_vs_TP")
pd.concat([fp_vs_tn, fn_vs_tp]).to_csv(OUT / "phase6_error_contrasts.csv", index=False)

print("\n\nFALSE POSITIVES vs TRUE NEGATIVES  (what makes a control look like anxiety) -- top 10:")
print(fp_vs_tn.head(10).round(4).to_string(index=False))
print("\n\nFALSE NEGATIVES vs TRUE POSITIVES  (what makes a case look like a control) -- top 10:")
print(fn_vs_tp.head(10).round(4).to_string(index=False))
print(f"\n  FN group size at this threshold: {int((pv_all.error_type=='FN').sum())} patients. "
      "Small n -> read FN contrasts as suggestive only.")
print(f"\nwrote {OUT/'phase6_characteristic_tests.csv'} and {OUT/'phase6_error_contrasts.csv'}")

In [ ]:
# ---------------------------------------------------------------------------
# 9b. The analysis table the supervisor drew -- generated, not transcribed.
#     Every row is a real misclassified patient with its measured
#     characteristics. Excerpts are printed for manual reading.
#     KEEP MIMIC TEXT INSIDE THIS SESSION.
# ---------------------------------------------------------------------------
N_SHOW = 8
flags = ["has_anx_term_seen", "has_indirect_term_seen", "has_psych_term_seen",
         "has_med_term_seen", "anx_all_negated", "anx_only_in_family_or_social",
         "anxiety_in_full_but_not_in_window", "truncated_by_512"]

def describe_note(r):
    tags = [f for f in flags if f in r.index and bool(r[f])]
    return "; ".join(t.replace("_", " ") for t in tags) or "none of the measured flags"

tbl = []
for et in ("FN", "FP"):
    sub = notes[notes.error_type == et]
    sub = (sub.nsmallest(N_SHOW, "p_anxiety_note") if et == "FN"
           else sub.nlargest(N_SHOW, "p_anxiety_note"))
    for _, r in sub.iterrows():
        tbl.append({"Error": et,
                    "Actual": "Anxiety" if r["label"] == 1 else "Control",
                    "Prediction": "Anxiety" if r["note_pred_label"] == 1 else "Control",
                    "p_note": round(float(r["p_anxiety_note"]), 3),
                    "chars": int(r["n_chars_full"]),
                    "dt_days": round(float(r["days_in_pkl"]), 1),
                    "Note characteristics (measured)": describe_note(r)})
analysis_table = pd.DataFrame(tbl)
print(analysis_table.to_string(index=False))
analysis_table.to_csv(OUT / "phase6_misclassified_examples.csv", index=False)

print("\n" + "=" * 78)
print("EXCERPTS FOR MANUAL READING -- do not copy out of this session (MIMIC DUA)")
print("=" * 78)
for et in ("FN", "FP"):
    sub = notes[notes.error_type == et]
    sub = (sub.nsmallest(3, "p_anxiety_note") if et == "FN"
           else sub.nlargest(3, "p_anxiety_note"))
    for _, r in sub.iterrows():
        print(f"\n--- {et}  patient {r['patient_id']}  note {r['note_id']}  "
              f"p_note={r['p_anxiety_note']:.3f}  dt={r['days_in_pkl']:.0f}d")
        print(f"    flags: {describe_note(r)}")
        print("    " + (r["text_model_saw"] or "")[:600].replace("\n", " ") + " ...")

## 10. Extract ClinicalBERT embeddings

The part of the meeting that got a straight "no". Now it gets done.

`ClinicalEmbedder` is two stages, and they are separated here because they can fail differently:

```text
Bio_ClinicalBERT -> [CLS] per chunk -> mean over chunks   ->  pooled, 768-d   (encoder output)
                                       -> Linear+GELU+Dropout+LayerNorm       ->  projection, 256-d
```

The 256-d projection is **the space prototypes live in** — `build_prototype` and `classify` both
operate there. The 768-d pooled `[CLS]` is what the fine-tuned encoder produced before the head.
If the classes separate at 768-d but not at 256-d, the projection is destroying signal; if neither
separates, the encoder is the constraint. Those are different repairs, so both are measured.

Note these are the **fine-tuned** encoder's embeddings from `best.pt`, not frozen off-the-shelf
Bio_ClinicalBERT. The frozen-encoder comparison already exists in the repo as the `bert_probe`
shallow baseline.

Same run, same frozen test plan, same episodes verified in section 1. The geometry pass below also
collects, for every queried note: distance to each class prototype, the support weights of its
episode, and the **uniform-weight counterfactual** — which is what sections 14–15 need.

In [ ]:
# ---------------------------------------------------------------------------
# 10.  embeddings + prototype geometry, one pass over the frozen test plan
# ---------------------------------------------------------------------------
test_plan = EpisodePlan.load(PLAN_DIR / f"{STEM}_test_k{K}.json")
geo, row_of, POOLED, PROJ = score_plan_cached(
    model, test_store, test_plan, want_geometry=True, desc="geometry")

chk2 = pool_to_patient(geo).merge(frozen, on="patient_id", suffixes=("_geo", "_csv"))
print(f"geometry pass reproduces predictions_test.csv: "
      f"max abs diff {float((chk2.p_anxiety_geo - chk2.p_anxiety_csv).abs().max()):.3e}")

# one row per queried note, carrying its verdict
gnote = (geo.groupby(["record_index", "note_id", "patient_id", "label"], as_index=False)
         .agg(d_proto_own=("d_proto_own", "mean"),
              d_proto_other=("d_proto_other", "mean"),
              d_proto_case=("d_proto_case", "mean"),
              d_proto_control=("d_proto_control", "mean"),
              margin=("margin_case_minus_control", "mean"),
              p_note=("p_anxiety_note", "mean"),
              p_note_uniform_w=("p_anxiety_note_uniform_w", "mean"),
              flip_rate=("decision_flips_without_weighting", "mean"),
              ep_H_norm=("ep_H_norm", "mean"),
              ep_weight_cv=("ep_weight_cv", "mean"),
              ep_weight_max_over_min=("ep_weight_max_over_min", "mean"),
              ep_corr_weight_vs_days=("ep_corr_weight_vs_days", "mean"),
              row_of=("row_of", "first")))
gnote = gnote.merge(err[["patient_id", "error_type", "correct"]], on="patient_id", how="left")
gnote["closer_to_wrong_prototype"] = gnote["d_proto_own"] > gnote["d_proto_other"]

E_PROJ = PROJ[gnote["row_of"].values]
E_POOL = POOLED[gnote["row_of"].values]
Y = gnote["label"].values.astype(int)
print(f"\nembeddings extracted for {len(gnote):,} queried test notes")
print(f"   pooled [CLS] (pre-projection) : {E_POOL.shape}")
print(f"   projection (prototype space)  : {E_PROJ.shape}")
np.save(OUT / "embeddings_projection.npy", E_PROJ)
gnote.to_csv(OUT / "phase6_note_geometry.csv", index=False)
print(f"wrote {OUT/'phase6_note_geometry.csv'}")

## 11. Measure embedding separation

*"Are the two classes actually separated in embedding space?"*

Three measurements, and the first one is the one that matters:

1. **Cross-validated centroid score.** Class centroids are built on 4/5 of the notes and the
   held-out 1/5 is scored by `cos(z, c_case) − cos(z, c_control)`; AUROC over all held-out folds.
   Centroids are held out because computing them on the same points they score is circular — that
   version would look better than the representation deserves. This is a **no-training** read of
   how much class information the geometry carries, and it is the right comparator for the model's
   0.7379: a prototypical network at inference is doing a weighted version of exactly this.
2. **Silhouette** with the true label as the cluster assignment: is there any global cluster
   structure aligned with the label at all? Values near 0 mean the classes are interleaved.
3. **k-NN label purity** (leave-one-out, k = 10): local structure. Reported against the
   **prevalence floor** (0.594 case) so it cannot be read as impressive when it is just the
   majority class.

Both spaces are measured — 768-d encoder output and 256-d projection.

The decision this feeds: if the centroid AUROC is close to the model's AUROC, the model is already
extracting most of what the representation contains, and no amount of prototype-weighting
engineering will move it. That is the embedding root cause.

In [ ]:
# ---------------------------------------------------------------------------
# 11.  separation of the two classes in each embedding space
# ---------------------------------------------------------------------------
from sklearn.metrics import roc_auc_score, silhouette_score
from sklearn.model_selection import StratifiedKFold
from sklearn.neighbors import NearestNeighbors

def l2(X):
    return X / np.maximum(np.linalg.norm(X, axis=1, keepdims=True), 1e-12)

def centroid_auroc(X, y, folds=5, seed=42):
    Xn = l2(X); scores = np.zeros(len(y))
    skf = StratifiedKFold(n_splits=folds, shuffle=True, random_state=seed)
    for tr, te in skf.split(Xn, y):
        c1 = l2(Xn[tr][y[tr] == 1].mean(0, keepdims=True))
        c0 = l2(Xn[tr][y[tr] == 0].mean(0, keepdims=True))
        scores[te] = (Xn[te] @ c1.T - Xn[te] @ c0.T).ravel()
    return float(roc_auc_score(y, scores)), scores

def knn_purity(X, y, k=10):
    Xn = l2(X)
    nn = NearestNeighbors(n_neighbors=k + 1, metric="cosine").fit(Xn)
    _, idx = nn.kneighbors(Xn)
    idx = idx[:, 1:]
    same = (y[idx] == y[:, None]).mean(1)
    frac_case = (y[idx] == 1).mean(1)
    return float(same.mean()), float(roc_auc_score(y, frac_case)), same, frac_case

SEP = {}
for name, X in (("pooled_cls_768", E_POOL), ("projection_256", E_PROJ)):
    auc_c, sc = centroid_auroc(X, Y)
    sil = float(silhouette_score(l2(X), Y, metric="cosine",
                                 sample_size=min(4000, len(Y)), random_state=42))
    pur, auc_knn, same, frac_case = knn_purity(X, Y, k=10)
    SEP[name] = {"centroid_auroc_cv": auc_c, "silhouette_by_label": sil,
                 "knn10_label_purity": pur, "knn10_frac_case_auroc": auc_knn}
    if name == "projection_256":
        gnote["knn10_same_label_frac"] = same
        gnote["knn10_frac_case"] = frac_case
        gnote["centroid_score"] = sc

sep = pd.DataFrame(SEP).T
MODEL_AUROC_TEST = float(roc_auc_score(frozen["label"].values, frozen["p_anxiety"].values))
PREVALENCE = float(frozen["label"].mean())
print(f"model AUROC on this frozen test set (patient level) : {MODEL_AUROC_TEST:.4f}")
print(f"case prevalence (the k-NN purity floor)             : {PREVALENCE:.4f}")
print()
print(sep.round(4).to_string())
sep.to_csv(OUT / "phase6_embedding_separation.csv")

E_SEP = SEP["projection_256"]["centroid_auroc_cv"]
print(f"\nheadroom = model AUROC - untrained centroid AUROC in prototype space "
      f"= {MODEL_AUROC_TEST:.4f} - {E_SEP:.4f} = {MODEL_AUROC_TEST - E_SEP:+.4f}")
print("  small or negative -> the episodic machinery is already extracting essentially")
print("     everything the representation holds; the REPRESENTATION is the ceiling.")
print("  large             -> the representation holds more than the model is using;")
print("     the head / prototype stage is the ceiling.")
print("\nNote: the note-level embedding AUROC and the patient-level model AUROC are")
print("not on identical units (patients pool ~3.6 episodes). Treat the comparison as")
print("indicative, and read it together with sections 13-15.")

## 12. Visualize embedding space

*"The goal is not a pretty plot. The goal is: does the embedding representation explain the
errors?"*

So the plots are read as diagnostics, not decoration:

* **by class** — the supervisor's two sketches. Two separated clouds, or interleaved points.
* **by error type** — where do FP and FN sit? Scattered through the wrong class's territory, or
  concentrated along the boundary?

PCA is shown first because it is linear and honest about global geometry. t-SNE is shown second, on
a subsample, with the standard warning attached: it manufactures apparent clusters and distances
between t-SNE clusters mean nothing. Neither plot is evidence on its own — the numbers in sections
11 and 13 are the evidence.

In [ ]:
# ---------------------------------------------------------------------------
# 12.  visualisation -- diagnostic, not decorative
# ---------------------------------------------------------------------------
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

Xn = l2(E_PROJ)
pca = PCA(n_components=2, random_state=42)
P2 = pca.fit_transform(Xn)
print(f"PCA explained variance (2 comps): {pca.explained_variance_ratio_.sum():.3f}")

rng = np.random.default_rng(42)
sub = rng.choice(len(Xn), size=min(3000, len(Xn)), replace=False)
T2 = TSNE(n_components=2, perplexity=30, init="pca", random_state=42,
          learning_rate="auto").fit_transform(Xn[sub])

COLORS = {"TP": "#2c7fb8", "TN": "#41ab5d", "FP": "#e6550d", "FN": "#d62728"}
fig, ax = plt.subplots(2, 2, figsize=(14, 12))

for a, (X2, ix, nm) in zip(ax[:, 0], [(P2, np.arange(len(Xn)), "PCA"),
                                      (T2, sub, "t-SNE")]):
    for lab, col, nmc in ((0, "#41ab5d", "control"), (1, "#2c7fb8", "anxiety")):
        m = Y[ix] == lab
        a.scatter(X2[m, 0], X2[m, 1], s=4, alpha=.35, c=col, label=nmc)
    a.set_title(f"{nm} — by TRUE class ({'projection 256-d'})"); a.legend(markerscale=3)

for a, (X2, ix, nm) in zip(ax[:, 1], [(P2, np.arange(len(Xn)), "PCA"),
                                      (T2, sub, "t-SNE")]):
    et = gnote["error_type"].values[ix]
    for t in ("TN", "TP", "FP", "FN"):
        m = et == t
        a.scatter(X2[m, 0], X2[m, 1], s=(6 if t in ("FP", "FN") else 3),
                  alpha=(.75 if t in ("FP", "FN") else .18), c=COLORS[t], label=t)
    a.set_title(f"{nm} — by ERROR type (FP/FN emphasised)"); a.legend(markerscale=3)

plt.tight_layout()
plt.savefig(OUT / "phase6_embedding_space.png", dpi=130)
plt.show()
print("t-SNE caveat: inter-cluster distances are meaningless and apparent clusters can be")
print("artefacts of perplexity. Sections 11 and 13 carry the evidence; this is orientation.")
print(f"wrote {OUT/'phase6_embedding_space.png'}")

## 13. Analyze FP/FN embedding neighborhoods

The four questions asked in the feedback, each turned into a number:

| question | measurement |
|---|---|
| Are FP/FN closer to the wrong class? | `closer_to_wrong_prototype`; sign of the centroid score |
| Are FP/FN near the class boundary? | `\|centroid_score\|` percentile within their own class |
| Are FP/FN embeddings unusual for their own class? | cosine to own-class centroid vs the class distribution |
| Do errors form clusters? | k-NN error-neighbour enrichment vs the base error rate |

The clustering test is the one that decides between a *data* story and a *boundary* story. If the
10 nearest neighbours of an error are themselves errors far more often than chance, the errors
occupy identifiable regions — a stratum that could be targeted. If error neighbours sit at the base
rate, the errors are spread along a boundary and there is no stratum to fix.

In [ ]:
# ---------------------------------------------------------------------------
# 13.  neighbourhood structure around the errors
# ---------------------------------------------------------------------------
Xn = l2(E_PROJ)
c1 = l2(Xn[Y == 1].mean(0, keepdims=True))
c0 = l2(Xn[Y == 0].mean(0, keepdims=True))
gnote["cos_to_case_centroid"]    = (Xn @ c1.T).ravel()
gnote["cos_to_control_centroid"] = (Xn @ c0.T).ravel()
gnote["cos_to_own_centroid"] = np.where(Y == 1, gnote["cos_to_case_centroid"],
                                        gnote["cos_to_control_centroid"])
gnote["centroid_pref_own"] = np.where(
    Y == 1, gnote["cos_to_case_centroid"] - gnote["cos_to_control_centroid"],
    gnote["cos_to_control_centroid"] - gnote["cos_to_case_centroid"])
gnote["boundary_proximity"] = -gnote["centroid_score"].abs()

print("Q1  Are FP/FN closer to the WRONG class?")
q1 = gnote.groupby("error_type")[["closer_to_wrong_prototype", "centroid_pref_own"]].mean()
print(q1.reindex(["TP", "TN", "FP", "FN"]).round(4).to_string())
print("     centroid_pref_own < 0 means the note sits nearer the OTHER class centroid.")

print("\nQ2  Are FP/FN near the boundary?  (|centroid score| — lower = nearer the boundary)")
q2 = gnote.groupby("error_type")["centroid_score"].apply(lambda s: s.abs().median())
print(q2.reindex(["TP", "TN", "FP", "FN"]).round(4).to_string())
for et, ref in (("FP", "TN"), ("FN", "TP")):
    a = gnote.loc[gnote.error_type == et, "centroid_score"].abs()
    b = gnote.loc[gnote.error_type == ref, "centroid_score"].abs()
    d, p = cliffs_delta(a, b)
    print(f"     |score| {et} vs {ref}: cliffs delta {d:+.3f}  p {p:.3g} "
          f"(negative = {et} nearer the boundary)")

print("\nQ3  Are FP/FN unusual for their own class?  (cosine to own-class centroid)")
q3 = gnote.groupby("error_type")["cos_to_own_centroid"].describe()[["mean", "50%", "std"]]
print(q3.reindex(["TP", "TN", "FP", "FN"]).round(4).to_string())

print("\nQ4  Do errors CLUSTER?")
is_err = (~gnote["correct"].values).astype(int)
nn = NearestNeighbors(n_neighbors=11, metric="cosine").fit(Xn)
_, idx = nn.kneighbors(Xn); idx = idx[:, 1:]
gnote["knn10_error_frac"] = is_err[idx].mean(1)
base = float(is_err.mean())
among_err = float(gnote.loc[~gnote["correct"], "knn10_error_frac"].mean())
among_ok  = float(gnote.loc[gnote["correct"], "knn10_error_frac"].mean())
enrich = among_err / max(base, 1e-9)
print(f"     base error rate among queried notes        : {base:.4f}")
print(f"     mean error fraction in an ERROR's 10-NN    : {among_err:.4f}  "
      f"(enrichment x{enrich:.2f})")
print(f"     mean error fraction in a CORRECT note's 10-NN: {among_ok:.4f}")
ERROR_CLUSTERING = enrich
print(f"     k-NN error-fraction AUROC for predicting error: "
      f"{roc_auc_score(is_err, gnote['knn10_error_frac']):.4f}")
print("\n     enrichment >= 1.5 -> errors occupy identifiable regions (a stratum exists)")
print("     enrichment ~  1.0 -> errors are spread along the boundary (no stratum)")

gnote.to_csv(OUT / "phase6_note_geometry.csv", index=False)

## 14. Inspect prototype distances

```text
query embedding -> support embeddings -> support weights -> class prototypes
                -> distance to each prototype -> prediction
```

Stored for every queried note, so the sentence the feedback asks for can be written from data:

> "This type of note was classified incorrectly because its embedding was closer to the control
> prototype despite being labelled as anxiety."

One thing to keep straight when reading the table: the **patient** verdict comes from the pooled
probability against the locked threshold, while `d_proto_own` / `d_proto_other` are **per-episode
geometry** averaged over the ~3.6 episodes a note appeared in. They are related but not identical,
and `closer_to_wrong_prototype` being below 100% inside the FP/FN groups is expected — a patient
can be misclassified at threshold 0.269 while still sitting marginally nearer its own prototype.

Note also that prototypes are rebuilt from a **different random support set in every episode**, so
per-episode variance in a note's score is itself a diagnostic: high variance means the prediction
depends on which 5 support patients were drawn, which is a support-set-composition problem, not a
note problem.

In [ ]:
# ---------------------------------------------------------------------------
# 14.  prototype geometry per error class
# ---------------------------------------------------------------------------
PG = ["d_proto_own", "d_proto_other", "margin", "p_note", "p_note_uniform_w"]
print("prototype geometry by patient-level verdict (mean):")
print(gnote.groupby("error_type")[PG + ["closer_to_wrong_prototype"]].mean()
      .reindex(["TP", "TN", "FP", "FN"]).round(4).to_string())
print("\n  d_proto_* is squared Euclidean on L2-normalised embeddings: 0 = identical, 4 = antipodal.")
print("  margin = d(control prototype) - d(case prototype); positive favours 'anxiety'.")

for et, ref in (("FP", "TN"), ("FN", "TP")):
    a = gnote.loc[gnote.error_type == et, "d_proto_own"]
    b = gnote.loc[gnote.error_type == ref, "d_proto_own"]
    d, p = cliffs_delta(a, b)
    print(f"  d_proto_own {et} vs {ref}: cliffs delta {d:+.3f}  p {p:.3g}")

# per-episode instability of a note's score
inst = (geo.groupby("record_index")["p_anxiety_note"].std()
        .rename("p_note_sd_across_episodes").reset_index())
gnote = gnote.merge(inst, on="record_index", how="left")
print("\nvariability of a note's score ACROSS episodes (different support sets):")
print(gnote.groupby("error_type")["p_note_sd_across_episodes"].mean()
      .reindex(["TP", "TN", "FP", "FN"]).round(4).to_string())
print(f"  overall mean within-note SD: {gnote['p_note_sd_across_episodes'].mean():.4f}")
print("  large relative to the gap between class means -> the support-set draw, not the")
print("  note, is driving individual predictions.")
print(f"  gap between class mean scores: "
      f"{abs(gnote.groupby('label')['p_note'].mean().diff().iloc[-1]):.4f}")

print("\nRepresentative errors, stated as the feedback asks:")
for et in ("FN", "FP"):
    sub = gnote[gnote.error_type == et]
    if sub.empty:
        continue
    sub = sub.nlargest(3, "d_proto_own") if et == "FN" else sub.nlargest(3, "margin")
    for _, r in sub.iterrows():
        truth = "anxiety" if r["label"] == 1 else "control"
        near = "case" if r["d_proto_case"] < r["d_proto_control"] else "control"
        print(f"  note {r['note_id']} (patient {r['patient_id']}, labelled {truth}): "
              f"embedding sat closer to the {near} prototype "
              f"(d_case={r['d_proto_case']:.3f}, d_control={r['d_proto_control']:.3f}), "
              f"p={r['p_note']:.3f}")

## 15. Inspect support weights for errors

Phase 4 measured the weights over episodes in aggregate. This restricts the same quantities to the
episodes where the query was **wrong**, and adds the measurement Phase 4 could not make.

Phase 4's own docstring is explicit about its limit: *"It cannot tell you whether a better
weighting scheme would help. It only tells you whether the current one is active."* The counterfactual
below closes exactly that gap, using the construction from `counterfactual_prototype.py`: inside
**one** model, on **one** set of embeddings, build each prototype twice —

```text
p_weighted = sum_i w_i z_i      (the model's learned w^T x w^C)
p_uniform  = (1/K) sum_i z_i    (weighting switched off at inference)
```

Everything except the weights is held fixed, so any difference is caused by the weighting and
nothing else. This separates the mechanism's effect from the encoder's, which
`analyse_mechanisms.py --reference` cannot do (its cosine 0.567 mixes both, since the two
prototypes come from two separately trained models).

The number that decides the model branch of the diagnosis:

```text
AUROC(weighted) - AUROC(uniform weights), same run, same episodes, same embeddings
```

If that is ≈ 0, then w^T and w^C are active (Phase 4 proved they are) but **causally irrelevant to
discrimination**, and no amount of retuning λ, β, temperature, or consistency passes will change
the headline. If it is materially non-zero, the weighting is doing real work and the model branch
is live.

In [ ]:
# ---------------------------------------------------------------------------
# 15.  support weights in error episodes + the uniform-weight counterfactual
# ---------------------------------------------------------------------------
WCOLS = ["ep_H_norm", "ep_weight_cv", "ep_weight_max_over_min", "ep_corr_weight_vs_days"]
print(f"uniform weight at K={K} is {1/K:.4f}; H_norm = 1.000 means exactly uniform.\n")
print("support-weight statistics of the episodes each note was queried in:")
print(gnote.groupby("error_type")[WCOLS].mean()
      .reindex(["TP", "TN", "FP", "FN"]).round(5).to_string())

print("\ncorrect vs incorrect:")
for c in WCOLS:
    d, p = cliffs_delta(gnote.loc[~gnote.correct, c], gnote.loc[gnote.correct, c])
    print(f"  {c:<26} cliffs delta {d:+.4f}   p {p:.4g}")
print("\n  Near-zero deltas mean errors are NOT concentrated in unusually-weighted")
print("  episodes -- the weighting is not selecting the failures.")

# ---- the counterfactual -----------------------------------------------------
pat_w = pool_to_patient(geo)
pat_u = (geo.groupby("patient_id")
         .agg(label=("label", "first"), p_anxiety=("p_anxiety_note_uniform_w", "mean"))
         .reset_index().sort_values("patient_id"))
m = pat_w.merge(pat_u, on="patient_id", suffixes=("_w", "_u"))
AUROC_W = float(roc_auc_score(m["label_w"], m["p_anxiety_w"]))
AUROC_U = float(roc_auc_score(m["label_w"], m["p_anxiety_u"]))
DELTA_WEIGHTING = AUROC_W - AUROC_U
FLIP_ALL = float(geo["decision_flips_without_weighting"].mean())
FLIP_ERR = float(geo.merge(err[["patient_id", "correct"]], on="patient_id")
                 .query("~correct")["decision_flips_without_weighting"].mean())

print("\n" + "=" * 74)
print("UNIFORM-WEIGHT COUNTERFACTUAL  (same model, same embeddings, weights -> 1/K)")
print("=" * 74)
print(f"  patient AUROC, learned weights      : {AUROC_W:.4f}")
print(f"  patient AUROC, uniform weights      : {AUROC_U:.4f}")
print(f"  DELTA attributable to the weighting : {DELTA_WEIGHTING:+.4f}")
print(f"  query decisions that flip, all      : {FLIP_ALL:.4%}")
print(f"  query decisions that flip, errors   : {FLIP_ERR:.4%}")
print(f"\n  For context, Phase 4 established the weights ARE non-uniform for this run:")
print(f"    H_norm 0.943, weight CV 0.282, max/min 19.38, corr(w, days) -0.631")
print(f"  Observed here on these episodes: H_norm {gnote.ep_H_norm.mean():.5f}, "
      f"CV {gnote.ep_weight_cv.mean():.5f}, max/min {gnote.ep_weight_max_over_min.mean():.3f}, "
      f"corr {gnote.ep_corr_weight_vs_days.mean():+.4f}")

json.dump({"auroc_weighted": AUROC_W, "auroc_uniform_weights": AUROC_U,
           "delta_attributable_to_weighting": DELTA_WEIGHTING,
           "decision_flip_rate_all": FLIP_ALL,
           "decision_flip_rate_errors": FLIP_ERR},
          open(OUT / "phase6_weighting_counterfactual.json", "w"), indent=2)
print(f"\nwrote {OUT/'phase6_weighting_counterfactual.json'}")

## 16. Determine likely root cause

*"Only after the above."*

The thresholds below are **fixed before the numbers are read** — that is the whole point of writing
them into the notebook rather than deciding afterwards which story the data supports. All three
branches are evaluated and all three verdicts are printed, including "no branch fired".

| branch | fires when | source |
|---|---|---|
| **DATA** | a candidate stratum exists from section 9 (Holm p < 0.05 **and** effect above the practical floor at patient level) **and** that stratum is underrepresented in train relative to test (ratio < 0.8) **and** errors cluster (k-NN enrichment ≥ 1.5) | §14 of the feedback: *"Which error pattern? How many training examples? Is the pattern underrepresented?"* |
| **EMBEDDING** | untrained cross-validated centroid AUROC in prototype space is within 0.03 of the trained model's AUROC, **and** both are below 0.80 | §15: *"heavy overlap ... ClinicalBERT's representation isn't sufficiently discriminative"* |
| **MODEL** | \|AUROC(weighted) − AUROC(uniform)\| ≥ 0.010 **or** the error-episode decision flip rate ≥ 5% | §16: *"embeddings separate reasonably well but prototype formation moves the representation incorrectly"* |

The DATA branch needs the train-split prevalence of the candidate stratum, so that is computed here
from the train rows of the cohort CSV — the same measurement instruments from section 8, applied to
the text the model saw during training.

**"No branch fires" is a legitimate outcome**, and given a 46% -lexical signal and a
0.7379 / 0.6284 blinding gap it is a live possibility. It corresponds to the third option the
feedback names: *"or simply a difficult/overlapping classification problem."* If that is the
verdict, the honest paper claim is the negative result plus the leakage-controlled framework — not
a manufactured intervention.

In [ ]:
# ---------------------------------------------------------------------------
# 16a. Train-split coverage of the candidate strata (the DATA branch needs it).
# ---------------------------------------------------------------------------
COVERAGE = {}
if CANDIDATES:
    print(f"candidate strata from section 9: {CANDIDATES}")
    text_flags = [c for c in CANDIDATES if c in TEXT_BIN]
    if text_flags:
        train_rows = cohort[cohort["split"] == "train"]
        val_rows = cohort[cohort["split"] == "val"]
        print(f"\nmeasuring the same flags on train ({len(train_rows):,} notes) "
              f"and val ({len(val_rows):,} notes) ...")
        def flag_rates(df, cols):
            out = {}
            txt = df["text"].fillna("").astype(str)
            # the model saw the first 512 wordpieces; approximate with a char cap
            # calibrated on the test split so train/test are measured comparably
            cap = int(np.median(notes["text_model_saw"].str.len()))
            txt = txt.str.slice(0, cap)
            for c in cols:
                if c == "has_anx_term_seen":       out[c] = float(txt.str.contains(P_ANX).mean())
                elif c == "has_med_term_seen":     out[c] = float(txt.str.contains(P_MED).mean())
                elif c == "has_psych_term_seen":   out[c] = float(txt.str.contains(P_PSY).mean())
                elif c == "has_indirect_term_seen":out[c] = float(txt.str.contains(P_IND).mean())
                elif c == "indirect_only_no_direct":
                    out[c] = float((txt.str.contains(P_IND) & ~txt.str.contains(P_ANX)).mean())
                else:                              out[c] = np.nan
            return out
        cov = pd.DataFrame({"train": flag_rates(train_rows, text_flags),
                            "val": flag_rates(val_rows, text_flags),
                            "test": {c: float(notes[c].mean()) for c in text_flags}})
        cov["train_over_test"] = cov["train"] / cov["test"].replace(0, np.nan)
        print(cov.round(4).to_string())
        COVERAGE = cov.to_dict()
        UNDERREPRESENTED = bool((cov["train_over_test"] < 0.8).any())
    else:
        print("candidate strata are not text flags; train coverage not computed for them.")
        UNDERREPRESENTED = False
else:
    print("no candidate strata from section 9 -> the DATA branch cannot fire on a stratum.")
    UNDERREPRESENTED = False
print(f"\nunderrepresented in train (ratio < 0.8): {UNDERREPRESENTED}")

In [ ]:
# ---------------------------------------------------------------------------
# 16b. Apply the pre-registered decision rules. No post-hoc threshold picking.
# ---------------------------------------------------------------------------
DATA_FIRES = bool(CANDIDATES) and UNDERREPRESENTED and (ERROR_CLUSTERING >= 1.5)
EMBEDDING_FIRES = (abs(MODEL_AUROC_TEST - E_SEP) <= 0.03) and (MODEL_AUROC_TEST < 0.80)
MODEL_FIRES = (abs(DELTA_WEIGHTING) >= 0.010) or (FLIP_ERR >= 0.05)

verdict = {
    "run": RUN_NAME,
    "evidence": {
        "candidate_strata": CANDIDATES,
        "stratum_underrepresented_in_train": UNDERREPRESENTED,
        "knn_error_clustering_enrichment": round(float(ERROR_CLUSTERING), 4),
        "model_auroc_test": round(MODEL_AUROC_TEST, 4),
        "untrained_centroid_auroc_projection": round(E_SEP, 4),
        "headroom_model_minus_centroid": round(MODEL_AUROC_TEST - E_SEP, 4),
        "auroc_weighted_minus_uniform": round(DELTA_WEIGHTING, 4),
        "decision_flip_rate_errors": round(FLIP_ERR, 4),
    },
    "branches": {"DATA": DATA_FIRES, "EMBEDDING": EMBEDDING_FIRES, "MODEL": MODEL_FIRES},
}
fired = [k for k, v in verdict["branches"].items() if v]
verdict["fired"] = fired
verdict["conclusion"] = (
    "NO_SINGLE_ROOT_CAUSE_AT_PREREGISTERED_THRESHOLDS" if not fired
    else ("_AND_".join(fired) if len(fired) > 1 else fired[0]))

print("=" * 78)
print("ROOT-CAUSE DIAGNOSIS")
print("=" * 78)
for k, v in verdict["evidence"].items():
    print(f"  {k:<44} {v}")
print()
print(f"  DATA branch      : {'FIRES' if DATA_FIRES else 'does not fire'}")
print(f"     needs a candidate stratum, underrepresented in train, and error clustering >= 1.5")
print(f"  EMBEDDING branch : {'FIRES' if EMBEDDING_FIRES else 'does not fire'}")
print(f"     needs |model AUROC - untrained centroid AUROC| <= 0.03 and model AUROC < 0.80")
print(f"  MODEL branch     : {'FIRES' if MODEL_FIRES else 'does not fire'}")
print(f"     needs |AUROC(weighted) - AUROC(uniform)| >= 0.010 or error flip rate >= 5%")
print()
print(f"  VERDICT: {verdict['conclusion']}")
if not fired:
    print("\n  Read this as the third option in the feedback: not underfitting, not")
    print("  overfitting, but an intrinsically overlapping problem at this")
    print("  representation. Combined with the blinding result (0.7379 -> 0.6284,")
    print("  ~46% of the above-chance margin lexical), the defensible paper claim is")
    print("  the leakage-controlled framework plus the negative mechanism finding.")
    print("  Do NOT manufacture an intervention to reach 0.80.")

json.dump(verdict, open(OUT / "phase6_root_cause_verdict.json", "w"), indent=2)
print(f"\nwrote {OUT/'phase6_root_cause_verdict.json'}")

## 17. Form one hypothesis

One hypothesis. One change. Written down **before** anything is retrained, so it cannot be revised
after seeing the result.

The pre-registration below is generated from the verdict, and it fixes four things:

1. the hypothesis, in falsifiable form
2. the single change that tests it — not data *and* encoder *and* dropout *and* learning rate
3. the metric and the comparator (the frozen five-seed baseline already in the repo)
4. the decision rule, including what counts as a **negative** result

Two constraints carried over from the repo's own discipline, and both matter more than the
hypothesis itself:

* **Selection happens on validation.** `configs/aux_w*.yaml` already carry the note
  *"SELECT ON VALIDATION ONLY. The test set is locked."* The intervention is chosen and tuned on
  val; test is touched once, at the end.
* **Five seeds, paired, on the frozen plans.** A one-seed improvement is not a result — the Phase 3B
  table has `aux_only` ranging 0.7233–0.7432 across seeds, a spread of 0.0199, which is larger than
  every mechanism effect measured so far.

In [ ]:
# ---------------------------------------------------------------------------
# 17.  pre-registration, generated from the verdict
# ---------------------------------------------------------------------------
SEED_SPREAD = 0.0199   # aux_only across seeds 42-46 in Phase 3B: 0.7233 - 0.7432
MDE = 0.02             # the smallest delta worth calling an improvement here

TEMPLATES = {
"DATA": f'''HYPOTHESIS
  The errors concentrate in a note stratum ({', '.join(CANDIDATES) if CANDIDATES else 'n/a'})
  that is underrepresented in the training split. If that stratum is better
  covered during meta-training, discrimination on it improves.

THE ONE CHANGE
  Alter ONLY the training episode composition so the stratum is sampled at its
  test-split rate. Do not touch model.py, the encoder, the optimiser, or any
  hyper-parameter. Rebuild the TRAIN plan only; val and test plans stay frozen.''',

"EMBEDDING": f'''HYPOTHESIS
  The Bio_ClinicalBERT-derived representation, not the prototypical head, is the
  binding constraint: an untrained centroid rule in the projection space already
  scores {E_SEP:.4f} against the model's {MODEL_AUROC_TEST:.4f}. If the representation
  carries more class information than the current pooling/projection exposes,
  changing that stage alone should move AUROC.

THE ONE CHANGE
  Change ONE element of the representation stage. Ranked by cost:
    (a) max_chunks 1 -> 4 in tokenize_cohort.py, so notes past token 512 are seen
        (justified only if 'anxiety_in_full_but_not_in_window' was non-trivial), or
    (b) an alternative clinical encoder, or
    (c) the pooling strategy.
  Pick ONE. The weighting mechanisms stay exactly as they are.''',

"MODEL": f'''HYPOTHESIS
  The weighting changes discrimination materially
  (AUROC weighted - uniform = {DELTA_WEIGHTING:+.4f}, error flip rate {FLIP_ERR:.2%}),
  so prototype construction is where the errors are produced and a corrected
  weighting/support-composition should improve them.

THE ONE CHANGE
  Modify ONE of: support-set composition, the distance function, temperature
  initialisation, or consistency_passes. Everything else, including the data
  pipeline and the encoder, is untouched.''',

"NO_SINGLE_ROOT_CAUSE_AT_PREREGISTERED_THRESHOLDS": f'''HYPOTHESIS
  None of the three pre-registered branches fired. The evidence is consistent
  with an intrinsically overlapping problem at this representation rather than a
  fixable data, embedding, or weighting defect.

THE ONE CHANGE
  NONE. Do not retrain. The correct next action is to write up:
    - the leakage-controlled benchmark and its zero-leakage certificate,
    - the auxiliary-controlled ablation and the five-seed paired result,
    - the mechanism diagnostics (active weights, no discrimination gain),
    - the uniform-weight counterfactual measured in section 15,
    - the blinding result (0.7379 -> 0.6284 anxiety-blinded),
    - and the error analysis in this notebook as the explanation of WHY.
  Sections 18-20 stay unexecuted, and that is the honest outcome.''',
}

key = verdict["conclusion"] if verdict["conclusion"] in TEMPLATES else "MODEL"
body = TEMPLATES[key]

prereg = f'''# TC-WPN Phase 6 — pre-registration
run under investigation : {RUN_NAME}
frozen test AUROC       : {MODEL_AUROC_TEST:.4f}   (2,278 patients, threshold {THRESHOLD:.5f})
root-cause verdict      : {verdict['conclusion']}

{body}

METRIC AND COMPARATOR
  Primary  : patient-level AUROC on the frozen episode plans.
  Comparator: the Phase 3B five-seed benchmark already in the repo —
              aux_only 0.7371 +/- 0.0081, tcwpn_full 0.7377 +/- 0.0031.
  Secondary : PR-AUC, sensitivity, specificity, Brier, ECE; and the anxiety-blinded
              arm, because an improvement that vanishes under blinding is lexical.

DECISION RULE (fixed now, not after the result)
  Seeds        : 42, 43, 44, 45, 46 — the same five, on the same frozen plans.
  Selection    : validation only. Test is scored ONCE, after the val decision.
  Success      : paired mean delta AUROC >= +{MDE:.3f} vs the frozen baseline AND
                 paired DeLong / paired t p < 0.05 AND >= 4/5 seeds improved.
  Failure      : anything else. A delta below the {SEED_SPREAD:.4f} seed-to-seed spread
                 of the baseline is noise and will be reported as no effect.
  Either way   : the result goes in the paper. A negative result here is publishable;
                 a positive result obtained by trying many changes is not.

WHAT IS EXPLICITLY NOT ALLOWED
  - changing more than one thing at a time
  - re-selecting the threshold on test
  - dropping a seed that disagrees
  - reporting the best of several attempted interventions
  - pursuing 0.80 because the proposal mentioned it
'''
(OUT / "phase6_hypothesis.md").write_text(prereg)
print(prereg)
print(f"wrote {OUT/'phase6_hypothesis.md'}")

## 18. Only then modify the model/data

Guarded on purpose. The cell below refuses to proceed unless a verdict exists and the
pre-registration has been read and explicitly acknowledged.

*"I would not touch `model.py` yet."* — so this section does not, and neither should the next
session until sections 1–17 have actually been run on real outputs and discussed with the
supervisor.

If the verdict was `NO_SINGLE_ROOT_CAUSE_AT_PREREGISTERED_THRESHOLDS`, the correct action is to
stop here and write the paper. That is not a failure state.

In [ ]:
# ---------------------------------------------------------------------------
# 18.  guard. Nothing is modified until the diagnosis is in and acknowledged.
# ---------------------------------------------------------------------------
ACKNOWLEDGED_PREREGISTRATION = False   # set True only after reading phase6_hypothesis.md
INTERVENTION_DESCRIPTION = ""          # one sentence; the ONE change

vpath = OUT / "phase6_root_cause_verdict.json"
if not vpath.exists():
    raise SystemExit("no verdict on disk -- run sections 1-16 first")
V = json.load(open(vpath))
print(f"verdict on disk: {V['conclusion']}")

if V["conclusion"] == "NO_SINGLE_ROOT_CAUSE_AT_PREREGISTERED_THRESHOLDS":
    print("\nSTOP. No branch fired. The pre-registration says: do not retrain.")
    print("Take sections 1-17 to the supervisor before any model or data change.")
elif not ACKNOWLEDGED_PREREGISTRATION:
    print("\nBLOCKED. Read phase6_hypothesis.md, then set")
    print("ACKNOWLEDGED_PREREGISTRATION = True and fill in INTERVENTION_DESCRIPTION.")
    print("This guard exists to stop the 'change BERT -> change LR -> change dropout'")
    print("loop the supervisor warned against.")
elif not INTERVENTION_DESCRIPTION.strip():
    print("\nBLOCKED. Write the ONE change in INTERVENTION_DESCRIPTION first.")
else:
    print(f"\nPROCEED with exactly one change: {INTERVENTION_DESCRIPTION}")
    print("Change it in a NEW config file (configs/<name>.yaml) or a NEW data-prep")
    print("argument. Do not edit model.py, sampler.py, or an existing config -- the")
    print("frozen baseline must stay reproducible for the section 20 comparison.")

## 19. Retrain

One configuration, five seeds, the same frozen val/test plans, with the repo's existing
`scripts.train` — no new training code, so the comparison in section 20 is like-for-like.

`--stem`, `--pkl-dir`, `--plan-dir` must match the frozen benchmark exactly. If the intervention
changes the data pipeline (a new `--max-chunks`, for example), the pkl changes, therefore the store
fingerprint changes, therefore **the episode plans must be rebuilt from the new pkl** — and the
paired DeLong test in section 20 is then no longer valid against the old predictions. In that case
the baseline must be re-scored on the new plans too. That is expensive and it is the reason a
pipeline-level intervention needs a stronger justification than a head-level one.

In [ ]:
# ---------------------------------------------------------------------------
# 19.  retrain -- prints the exact commands; runs nothing unless RUN_TRAINING.
# ---------------------------------------------------------------------------
RUN_TRAINING = False
NEW_CONFIG   = "configs/<your_new_config>.yaml"
SEEDS        = [42, 43, 44, 45, 46]

cmds = []
for s in SEEDS:
    cmds.append(f"python -m scripts.train --config {NEW_CONFIG} --k {K} --seed {s} "
                f"--stem {STEM} --pkl-dir {PKL_DIR} --plan-dir {PLAN_DIR} "
                f"--results /kaggle/working/results")
    cmds.append(f"python -m scripts.evaluate "
                f"--run /kaggle/working/results/{STEM}/<config_name>_k{K}_seed{s} "
                f"--split test --pkl-dir {PKL_DIR} --plan-dir {PLAN_DIR} --bootstrap 2000")
print("EXACT COMMANDS (one config, five seeds, frozen plans):\n")
print("\n".join(cmds))
print("\nOn a T4 one seed of this configuration took ~47 minutes (from manifest.json),")
print("so five seeds is roughly 4 hours of the ~30 h/week quota. Run one configuration")
print("per session, as Phase 3B did.")

if RUN_TRAINING:
    for c in cmds:
        print("\n$", c)
        get_ipython().system(c)
else:
    print("\nRUN_TRAINING is False -- nothing was trained. Set it True only after section 18 clears.")

## 20. Evaluate against frozen baseline

The comparison is paired: both models are scored on byte-identical episodes from the same frozen
plans, which is precisely why `make_episode_plans.py` serialises plans to disk instead of sampling
live.

Two tests, both required:

* **Paired DeLong** on the seed-42 patient vectors, via the repo's own `scripts.compare_models pair`
* **Paired across seeds** — mean Δ, median Δ, seeds-improved, paired t and Wilcoxon, Cohen's dz —
  the same statistic Phase 4 used, because per-seed differences are the right unit when every
  configuration ran on the same five seeds

And the decision rule from section 17 is applied mechanically, not narratively. The relevant
comparator numbers, already in the repo:

```text
aux_only      0.7371 +/- 0.0081     (seed range 0.7233 - 0.7432)
temporal_aux  0.7377 +/- 0.0032
pcw_aux       0.7291 +/- 0.0083
tcwpn_full    0.7377 +/- 0.0031
```

In [ ]:
# ---------------------------------------------------------------------------
# 20.  paired comparison against the frozen baseline
# ---------------------------------------------------------------------------
NEW_RUN_DIR = None      # e.g. Path(f"/kaggle/working/results/{STEM}/<config>_k5_seed42")

if NEW_RUN_DIR is None:
    print("No new run to compare yet. What this cell will do once there is one:\n")
    print(f"  python -m scripts.compare_models pair \\")
    print(f"      --a <new_run>/predictions_test.csv \\")
    print(f"      --b {RUN_DIR}/predictions_test.csv")
    if BASE_DIR:
        print("\nDemonstrating the machinery on two runs that already exist "
              "(tcwpn_full vs aux_only, seed 42):")
        !python -m scripts.compare_models pair \
            --a {RUN_DIR}/predictions_test.csv \
            --b {BASE_DIR}/predictions_test.csv \
            --out {OUT}/phase6_pair_tcwpn_vs_aux.json
        print("\nThat is the frozen negative result this notebook set out to explain.")
else:
    !python -m scripts.compare_models pair \
        --a {NEW_RUN_DIR}/predictions_test.csv \
        --b {RUN_DIR}/predictions_test.csv \
        --out {OUT}/phase6_pair_new_vs_frozen.json

    BASELINE = {"aux_only": [0.7432, 0.7233, 0.7413, 0.7413, 0.7364],
                "tcwpn_full": [0.7379, 0.7394, 0.7386, 0.7403, 0.7323]}
    new_auroc = []      # fill with the five new-seed AUROCs from eval_test.json
    if len(new_auroc) == len(SEEDS):
        base = np.array(BASELINE["tcwpn_full"], float)
        new = np.array(new_auroc, float)
        d = new - base
        t_p = stats.ttest_rel(new, base).pvalue
        w_p = stats.wilcoxon(new, base).pvalue
        dz = d.mean() / d.std(ddof=1)
        print(f"\nmean delta {d.mean():+.4f}   median {np.median(d):+.4f}   "
              f"seeds better {int((d > 0).sum())}/{len(d)}   "
              f"paired t p {t_p:.4f}   wilcoxon p {w_p:.4f}   dz {dz:.3f}")
        success = (d.mean() >= MDE) and (t_p < 0.05) and ((d > 0).sum() >= 4)
        print(f"\nPRE-REGISTERED DECISION: {'SUCCESS' if success else 'NO EFFECT'}")
        print("Report it either way.")

## What to tell the supervisor, and what not to claim

**What was asked for, and where it is:**

| requirement from the meeting | section |
|---|---|
| Examine misclassified samples | 2 |
| Compare correct vs incorrect notes | 3–4, 9 |
| Identify characteristics of error cases | 5–8 |
| Inspect embeddings of errors | 10, 13 |
| Inspect embedding separation | 11–12 |
| Inspect prototype behaviour **for errors** | 14–15 |
| Determine data vs embedding vs model root cause | 16 |
| Make a targeted improvement based on the root cause | 17–19 (gated) |
| Re-evaluate after the targeted change | 20 (gated) |
| Training vs validation vs test performance | 0 |

**Do not write** that TC-WPN improves few-shot anxiety detection. One seed of five, median Δ
negative, p = 0.886.

**Do write**, once these cells have run on real outputs, the version supported by the numbers:

> The proposed temporal and prototype-consistency weighting mechanisms are demonstrably active
> (normalised entropy 0.943, max/min weight ratio 19.4, corr(w, days before index) −0.631), yet
> under auxiliary-controlled ablation across five seeds on a patient-disjoint, ICD-labelled cohort
> with a zero-leakage episode certificate they produce no improvement in discrimination over the
> auxiliary-controlled prototypical baseline (Δ AUROC = +0.0006, 1/5 seeds improved, p = 0.886).
> Error analysis localises the failure to [whatever section 16 actually reports], and a
> uniform-weight counterfactual within the same model shows the weighting accounts for
> [DELTA_WEIGHTING] AUROC.

**Also report the blinding gap**, which is the strongest caveat in the whole project: 0.7379 →
0.6284 when anxiety terms are deleted. Against the above-chance margin that is 0.2379 → 0.1284, so
roughly **46% of the signal is lexical**. Adding medication terms changed almost nothing
(0.6284 → 0.6291), so it is the anxiety vocabulary specifically.

**On the proposal's 80%:** the controlled result is 0.73–0.74. A five-seed, leakage-certified 0.737
is worth more than an isolated 0.80 with an uncertain provenance. If a root-caused intervention
reaches 0.80 honestly, excellent. If not, report 0.737.

---

### Files this notebook writes to `/kaggle/working/phase6/`

| file | contents |
|---|---|
| `error_analysis.csv` | patient_id, actual_label, predicted_probability, predicted_label, error_type, n_episodes |
| `note_error_table.csv` | one row per queried note with its verdict and metadata |
| `note_error_table_with_characteristics.csv` | the above plus every measured note characteristic |
| `phase6_split_performance.csv` | train / val / test metric bundle |
| `phase6_characteristic_tests.csv` | the full test family with Holm-adjusted p and effect sizes |
| `phase6_error_contrasts.csv` | FP-vs-TN and FN-vs-TP contrasts |
| `phase6_misclassified_examples.csv` | the analysis table, generated from real errors |
| `phase6_embedding_separation.csv` | centroid AUROC, silhouette, k-NN purity, both spaces |
| `phase6_note_geometry.csv` | prototype distances, support-weight stats, neighbourhood measures |
| `phase6_weighting_counterfactual.json` | AUROC weighted vs uniform, decision flip rates |
| `phase6_root_cause_verdict.json` | the pre-registered branch evaluation |
| `phase6_hypothesis.md` | the pre-registration for the next experiment |
| `phase6_embedding_space.png` | PCA and t-SNE, by class and by error type |

Commit the session so the next notebook can attach these as an input, exactly as Phase 3B → Phase 4
already does.

**Do not start by rebuilding TC-WPN. Start by dissecting the errors.**